# Radar target classification - v20 (live stream, physics gate, honest confidence)

**Input:** ICD packets (CSV / Excel). **Output:** a live per-packet verdict for every `track_no`, a final summary per track, and dashboards.
Run the cells top to bottom (about 6-10 min on CPU, much less on a GPU). Set your file names in **Cell 2** (`INPUT_FILES`).

## What was wrong with v11 / v12 / v19 (diagnosed by retraining them and testing on your two files)
| # | Problem | Evidence | Fix in this notebook |
|---|---|---|---|
| 1 | Network never saw short histories (trained on full 30-packet windows only); inference **repeated the first packet** to fill the window | `crossing_2.xlsx` has 2-3 packets per track -> v19 answered with **97 % confidence** | Trained on histories of every length (1, 2, 3, ...) with a valid-step mask; confidence is calibrated (temperature), status says PROVISIONAL |
| 2 | **No physics check**: nothing said a target at 1000 m or 5000 m cannot be a boat/ship | sky targets labelled with sea-ish classes | `ENVELOPE` feasibility gate multiplied into the belief (Cell 2, 5, 8) |
| 3 | Simulator fed **white-noise acceleration**; real ICD data is **tracker output** (smooth, lagging) | real straight flight has `accel` ~ 0.01 m/s^2, looked "ship-like" | Cell 4 simulates noisy measurements and runs a **Kalman tracker** that writes the ICD columns |
| 4 | v11 used `std_dev_*`/`cov_*`; the simulator tied them to the class (1400 -> ship) | real covariances are geometry/tracker dependent | Not classifier inputs any more; used only for diagnostics. Same for `range_mag`, absolute position |
| 5 | Disguise (breakout) logic had been lost | - | Cell 3: FIGHTER-as-CIVILIAN, DRONE-as-HELICOPTER, SURFACE_MISSILE-as-BOAT, AERIAL_MISSILE-as-FIGHTER; label = what the target *acts like now*; belief can switch (Cell 8) |
| 6 | Timestamps / axes in samples are inconsistent | `crossing_2`: dt 2.5 s vs 11 s while positions imply ~10 s; in both files vel X/Y look swapped vs pos X/Y | Cell 12 audit warns; classifier uses neither timestamps nor vector directions |

## About `std_dev_*` and `cov_*`
They are the tracker's own estimate of its error (1-sigma of position / velocity / acceleration, and position-velocity covariance).
They describe **measurement quality, not target type**, so they must not decide the class. We show them as track-quality diagnostics only.

## Cell map
2 settings + physics envelopes | 3-4 simulator (motion + tracker) | 5 file reader + features + gate | 6-7 data, network, calibration |
8-10 fusion, evaluation, plots | 11 live engine | 12 load + audit | 13 **live replay** | 14 **summary** | 15-16 **dashboards** | 17 self-tests | 18 save


## Cell 2 - Classes, physics envelopes and global settings
`ENVELOPE` is the physics rulebook (what each class can physically do). It is used in two places:
the simulator samples *inside* it, and the inference engine uses it as a feasibility gate so that e.g. a
target at 1000 m altitude can never be reported as a boat.

In [ ]:
import os, sys, json, math, time, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, accuracy_score
from scipy.optimize import minimize_scalar
from scipy.special import softmax, logsumexp

try:
    from IPython.display import display
except Exception:                                   # plain python fallback
    display = print

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

# ----------------------------- user settings -----------------------------
INPUT_FILES = ["radar_data.csv", "crossing_2.xlsx"]   # any csv/xlsx in the ICD format (one or many)
ALT_SCALE = 1.0          # multiply pos_z by this to get metres   (0.3048 if the ICD were in feet)
VEL_SCALE = 1.0          # multiply vel/accel by this to get m/s, m/s^2
SEQ_LEN = 24             # packets the network looks at (shorter histories are padded + masked)
TRACKS_PER_CLASS = dict(train=170, val=40, test_iid=40, test_stress=40)
MAX_EPOCHS = 25
BATCH_SIZE = 256
PLAYBACK_DELAY = 0.0     # seconds between printed packets (0 = instant; 0.05 = "live" feel)
PRINT_MODE = "every"     # "every" packet | "changes" only (class / status changes)
ARTIFACT_DIR = Path("radar_bundle_v20"); ARTIFACT_DIR.mkdir(exist_ok=True)

CLASSES = ["CIVILIAN_AIRCRAFT", "FIGHTER_AIRCRAFT", "HELICOPTER", "DRONE",
           "SURFACE_MISSILE", "AERIAL_MISSILE", "SKIFF", "BOAT", "SHIP"]
K = len(CLASSES)
CLS_ID = {c: i for i, c in enumerate(CLASSES)}
SURFACE = {"SKIFF", "BOAT", "SHIP"}
DOMAIN_OF = {c: ("SURFACE" if c in SURFACE else "AIR") for c in CLASSES}
AIR_IDX = np.array([i for i, c in enumerate(CLASSES) if c not in SURFACE])
SEA_IDX = np.array([i for i, c in enumerate(CLASSES) if c in SURFACE])
COLORS = {"CIVILIAN_AIRCRAFT": "#1f77b4", "FIGHTER_AIRCRAFT": "#d62728", "HELICOPTER": "#9467bd",
          "DRONE": "#ff7f0e", "SURFACE_MISSILE": "#8c564b", "AERIAL_MISSILE": "#e377c2",
          "SKIFF": "#2ca02c", "BOAT": "#17becf", "SHIP": "#7f7f7f", "UNDETERMINED": "#cccccc"}

# Physical feasibility limits (generous on purpose). alt [m], speed [m/s], |vz| [m/s], lateral load [g]
ENVELOPE = {
    "CIVILIAN_AIRCRAFT": dict(alt=(100, 15500), speed=(40, 340),  vz=30,  g=3.0),
    "FIGHTER_AIRCRAFT":  dict(alt=(0, 21000),   speed=(60, 1000), vz=300, g=12.0),
    "HELICOPTER":        dict(alt=(0, 6500),    speed=(0, 150),   vz=15,  g=3.0),
    "DRONE":             dict(alt=(0, 20000),   speed=(0, 260),   vz=40,  g=8.0),
    "SURFACE_MISSILE":   dict(alt=(0, 30000),   speed=(20, 1500), vz=300, g=15.0),
    "AERIAL_MISSILE":    dict(alt=(0, 30000),   speed=(60, 2000), vz=600, g=45.0),
    "SKIFF":             dict(alt=(0, 90),      speed=(0, 32),    vz=3,   g=1.0),
    "BOAT":              dict(alt=(0, 100),      speed=(0, 40),    vz=3,   g=0.8),
    "SHIP":              dict(alt=(0, 120),      speed=(0, 25),    vz=3,   g=0.3),
}
print("TensorFlow", tf.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))
print("Classes:", ", ".join(CLASSES))

## Cell 3 - Simulator, part A: realistic target motion
v11/v12 drew a fresh random acceleration at every step (white noise). Real targets fly **manoeuvres**:
straight legs, constant-rate turns, orbits, climbs, speed changes. Each class gets its own manoeuvre mix and
g-limits. Some classes start by *mimicking* another class (a disguise) and break out later:
FIGHTER looks like CIVILIAN, DRONE like HELICOPTER, SURFACE_MISSILE like BOAT, AERIAL_MISSILE like FIGHTER.

In [ ]:
G0 = 9.81
# speed=(initial range) clip=(hard speed limits) alt=(initial range) turn_g=(lateral g in turns) along=(max m/s^2 speed change)
SPEC = {
 "CIVILIAN_AIRCRAFT": dict(speed=(150, 290), clip=(60, 320), alt=(3000, 12500), vz=12, turn_g=(0.03, 0.35), along=0.6,
                           mix=dict(straight=.55, turn=.15, orbit=.12, climb=.12, speedchg=.06), dur=(60, 400)),
 "FIGHTER_AIRCRAFT":  dict(speed=(180, 700), clip=(90, 950), alt=(200, 16000), vz=120, turn_g=(0.5, 8.0), along=12.0,
                           mix=dict(straight=.25, turn=.35, orbit=.05, climb=.20, speedchg=.15), dur=(15, 120)),
 "HELICOPTER":        dict(speed=(0, 80), clip=(0, 120), alt=(20, 3000), vz=8, turn_g=(0.02, 0.4), along=1.5,
                           mix=dict(straight=.35, turn=.25, orbit=.10, climb=.15, speedchg=.15), dur=(20, 150)),
 "DRONE":             dict(speed=(8, 70), clip=(3, 200), alt=(30, 5000), vz=10, turn_g=(0.05, 1.5), along=3.0,
                           mix=dict(straight=.30, turn=.30, orbit=.15, climb=.10, speedchg=.15), dur=(15, 150)),
 "SURFACE_MISSILE":   dict(speed=(250, 900), clip=(150, 1300), alt=(5, 60), vz=25, turn_g=(0.3, 12.0), along=15.0,
                           mix=dict(straight=.55, turn=.20, orbit=.0, climb=.10, speedchg=.15), dur=(15, 80)),
 "AERIAL_MISSILE":    dict(speed=(400, 1300), clip=(250, 1800), alt=(1000, 22000), vz=300, turn_g=(1.0, 35.0), along=40.0,
                           mix=dict(straight=.45, turn=.30, orbit=.0, climb=.15, speedchg=.10), dur=(10, 60)),
 "SKIFF":             dict(speed=(3, 22), clip=(0.5, 30), alt=(0, 4), vz=0, turn_g=(0.02, 0.5), along=1.0,
                           mix=dict(straight=.35, turn=.40, orbit=.05, climb=.0, speedchg=.20), dur=(20, 120)),
 "BOAT":              dict(speed=(2, 16), clip=(0.5, 26), alt=(0, 8), vz=0, turn_g=(0.01, 0.25), along=0.4,
                           mix=dict(straight=.55, turn=.25, orbit=.05, climb=.0, speedchg=.15), dur=(40, 300)),
 "SHIP":              dict(speed=(1, 14), clip=(0.3, 18), alt=(5, 28), vz=0, turn_g=(0.002, 0.04), along=0.08,
                           mix=dict(straight=.78, turn=.12, orbit=.02, climb=.0, speedchg=.08), dur=(120, 600)),
}
MIMIC = {"FIGHTER_AIRCRAFT": "CIVILIAN_AIRCRAFT", "DRONE": "HELICOPTER",
         "SURFACE_MISSILE": "BOAT", "AERIAL_MISSILE": "FIGHTER_AIRCRAFT"}
P_DISGUISE = 0.30
TRUTH_DT = 2.0           # seconds, integration step of the true motion


def _pick(rng, mix):
    names = list(mix); p = np.array([mix[k] for k in names], float)
    return names[int(rng.choice(len(names), p=p / p.sum()))]


def simulate_truth(cls, rng, total_s, mimic=None, breakout_s=None):
    """True motion on a TRUTH_DT grid. Returns t, pos(n,3), vel(n,3), behaviour(n) (class that is *acting* now)."""
    n = int(total_s / TRUTH_DT) + 2
    t = np.arange(n) * TRUTH_DT
    sea = cls in SURFACE
    spec = SPEC[mimic] if mimic else SPEC[cls]
    spec_true = SPEC[cls]
    beh = np.empty(n, dtype=object)

    def init(sp):
        s0 = rng.uniform(*sp["speed"]); a0 = rng.uniform(*sp["alt"])
        return s0, a0
    speed, alt = init(spec)
    if mimic == "SURFACE_MISSILE" or (mimic == "BOAT"):          # launch phase: low and slow
        alt = rng.uniform(0, 25)
    if cls == "SURFACE_MISSILE" and not mimic and rng.random() < 0.3:
        alt = rng.uniform(300, 7000)                              # lofted trajectory
    radius = rng.uniform(15e3, 150e3); bearing = rng.uniform(-np.pi, np.pi)
    pos = np.array([radius * np.cos(bearing), radius * np.sin(bearing), alt])
    heading = rng.uniform(-np.pi, np.pi)
    vz = 0.0
    omega = 0.0; along = 0.0; vz_goal = 0.0; seg_left = 0.0
    P_out = np.zeros((n, 3)); V_out = np.zeros((n, 3))
    heave_amp = rng.uniform(0.2, 2.0) if sea else 0.0
    heave_per = rng.uniform(20, 60); heave_ph = rng.uniform(0, 2 * np.pi)
    speed_goal = None
    broke = False
    cur = spec
    cur_name = mimic if mimic else cls
    for i in range(n):
        ti = t[i]
        if mimic and (not broke) and ti >= breakout_s:         # DISGUISE BREAKOUT
            broke = True; cur = spec_true; cur_name = cls
            speed_goal = rng.uniform(*spec_true["speed"]) * rng.uniform(1.1, 1.6) if not sea else speed
            speed_goal = float(np.clip(speed_goal, *spec_true["clip"]))
            heading += rng.choice([-1, 1]) * rng.uniform(0.3, 1.2)
            if cls in ("SURFACE_MISSILE",):
                vz_goal = rng.uniform(5, 40)
            seg_left = rng.uniform(40, 90)
            omega = 0.0; along = 0.0
        beh[i] = cur_name
        if seg_left <= 0:
            kind = _pick(rng, cur["mix"])
            seg_left = rng.uniform(*cur["dur"])
            omega = 0.0; along = 0.0
            vz_goal = 0.0                                    # every new segment starts level unless it is a climb
            sp = max(speed, 1.0)
            if kind in ("turn", "orbit"):
                g = rng.uniform(*cur["turn_g"]) * (0.4 if kind == "orbit" else 1.0)
                omega = rng.choice([-1, 1]) * g * G0 / max(sp, 8.0)
                if kind == "orbit": seg_left *= 1.5
            elif kind == "speedchg":
                along = rng.choice([-1, 1]) * rng.uniform(0.15, 1.0) * cur["along"]
            elif kind == "climb" and not sea:
                sign = 1.0 if pos[2] < 300 else (-1.0 if pos[2] > 1.2 * cur["alt"][1] else rng.choice([-1.0, 1.0]))
                vz_goal = sign * rng.uniform(0.2, 1.0) * cur["vz"]
            elif kind == "straight":
                vz_goal = rng.normal(0, 0.02 * max(cur["vz"], 0.2)) if not sea else 0.0
        seg_left -= TRUTH_DT
        if speed_goal is not None:
            along = float(np.clip((speed_goal - speed) / 8.0, -cur["along"], cur["along"]))
            if abs(speed_goal - speed) < 0.02 * speed_goal: speed_goal = None
        heading += omega * TRUTH_DT
        speed = float(np.clip(speed + along * TRUTH_DT, *cur["clip"]))
        vz += (vz_goal - vz) * min(TRUTH_DT / 8.0, 1.0)
        lim = min(cur["vz"], 0.95 * speed)
        vz = float(np.clip(vz, -lim, lim))
        hs = math.sqrt(max(speed * speed - vz * vz, 0.0))
        vel = np.array([hs * math.cos(heading), hs * math.sin(heading), vz])
        pos = pos + vel * TRUTH_DT
        if not sea and pos[2] < 5.0 and vz < 0:                  # flat floor over the sea/ground
            pos[2] = 5.0; vz = 0.0; vz_goal = max(vz_goal, 0.0)
        if sea:
            ref = alt
            hv = heave_amp * 2 * np.pi / heave_per * math.cos(2 * np.pi * ti / heave_per + heave_ph)
            pos[2] = ref + heave_amp * math.sin(2 * np.pi * ti / heave_per + heave_ph)
            vel = np.array([vel[0], vel[1], hv])
        P_out[i] = pos; V_out[i] = vel
    return t, P_out, V_out, beh
print("Simulator part A ready")

## Cell 4 - Simulator, part B: a radar *tracker* in the loop (this is what makes the data look like your ICD)
Your CSV does not contain raw measurements: it contains the output of a **tracker** (a Kalman filter). That is why
`vel`/`accel` are smooth and lagging, and why `std_dev_*` / `cov_*` are the filter's own error estimates.
v11/v12 fed white-noise acceleration to the network, so real (smooth) data looked like a ship. Here we simulate
noisy position measurements, run a constant-acceleration Kalman filter and write out exactly the ICD columns.

Everything sensor-related is **randomised per track** (noise, filter tuning, scan period, missed detections) so the
network cannot lock onto one sensor setting.

In [ ]:
FAMILIES = {   # sensor "weather": noise multiplier range, detection probability range, scan periods [s]
    "nominal_train": dict(noise=(0.7, 1.3), pd=(0.85, 1.0), periods=[10.0]),
    "noisy_train":   dict(noise=(1.2, 2.5), pd=(0.60, 0.95), periods=[10.0, 5.0]),
    "mixed_train":   dict(noise=(0.7, 2.5), pd=(0.70, 1.0), periods=[10.0, 10.0, 5.0, 2.5]),
    "val_shift":     dict(noise=(1.5, 3.0), pd=(0.60, 0.95), periods=[10.0, 5.0, 2.5]),
    "test_iid":      dict(noise=(0.7, 2.5), pd=(0.70, 1.0), periods=[10.0, 10.0, 5.0, 2.5]),
    "test_stress":   dict(noise=(2.5, 5.0), pd=(0.50, 0.85), periods=[10.0, 5.0, 2.5]),
}
ICD_COLS = ["packet_size", "source", "destination", "packet_type", "packet_category", "radar_id", "track_no",
            "pos_x", "pos_y", "pos_z", "range_mag", "vel_x", "vel_y", "vel_z", "vel_mag", "heading",
            "accel_x", "accel_y", "accel_z", "accel_mag", "std_dev_x", "std_dev_y", "std_dev_z",
            "std_dev_vx", "std_dev_vy", "std_dev_vz", "std_dev_ax", "std_dev_ay", "std_dev_az",
            "cov_x_vx", "cov_y_vy", "cov_z_vz", "ntp_sec", "ntp_frac", "assoc_indication"]


def kalman_track(t_rep, P_true, V_true, rng, fam):
    """Run a 3-axis constant-acceleration Kalman filter on noisy position measurements."""
    n = len(t_rep)
    mult = rng.uniform(*fam["noise"])
    sig_r = np.exp(rng.uniform(np.log(15), np.log(150))) * mult
    sig_az = np.exp(rng.uniform(np.log(5e-4), np.log(5e-3))) * mult
    sig_z = np.exp(rng.uniform(np.log(3), np.log(45))) * math.sqrt(mult)
    q = np.exp(rng.uniform(np.log(2e-4), np.log(2e-2)))          # jerk std [m/s^3] (filter tuning)
    pdet = rng.uniform(*fam["pd"])
    inflate = np.exp(rng.uniform(0, np.log(30)))                  # trackers report pessimistic covariances
    x = np.zeros((3, 3)); Pm = np.zeros((3, 3, 3))
    out = np.zeros((n, 3, 3)); sd = np.zeros((n, 3, 3)); cv = np.zeros((n, 3)); upd = np.zeros(n, bool)
    for k in range(n):
        p = P_true[k]; v = V_true[k]
        rxy = max(math.hypot(p[0], p[1]), 1.0)
        u = np.array([p[0] / rxy, p[1] / rxy]); w = np.array([-u[1], u[0]])
        sc = rxy * sig_az
        noise_xy = u * rng.normal(0, sig_r) + w * rng.normal(0, sc)
        z_meas = np.array([p[0] + noise_xy[0], p[1] + noise_xy[1], p[2] + rng.normal(0, sig_z)])
        R = np.array([sig_r**2 * u[0]**2 + sc**2 * w[0]**2, sig_r**2 * u[1]**2 + sc**2 * w[1]**2, sig_z**2]) + 1.0
        if k == 0:
            sv = max(2.0, 0.03 * np.linalg.norm(v))
            x[:, 0] = z_meas; x[:, 1] = v + rng.normal(0, sv, 3); x[:, 2] = 0.0
            for a in range(3): Pm[a] = np.diag([R[a], sv**2, 4.0])
            upd[k] = True
        else:
            dt = t_rep[k] - t_rep[k - 1]
            F = np.array([[1, dt, 0.5 * dt * dt], [0, 1, dt], [0, 0, 1.0]])
            Q = (q**2) * np.array([[dt**5 / 20, dt**4 / 8, dt**3 / 6], [dt**4 / 8, dt**3 / 3, dt**2 / 2], [dt**3 / 6, dt**2 / 2, dt]])
            x = x @ F.T
            Pm = np.einsum("ij,ajk,lk->ail", F, Pm, F) + Q
            if rng.random() < pdet:
                upd[k] = True
                for a in range(3):
                    S = Pm[a, 0, 0] + R[a]; Kg = Pm[a, :, 0] / S
                    x[a] = x[a] + Kg * (z_meas[a] - x[a, 0])
                    Pm[a] = Pm[a] - np.outer(Kg, Pm[a, 0, :])
        out[k] = x
        sd[k] = np.sqrt(np.maximum(np.stack([np.diag(Pm[a]) for a in range(3)]), 0)) * inflate
        cv[k] = np.array([Pm[a, 0, 1] for a in range(3)]) * inflate**2
    return out, sd, cv, upd


def make_track(cls, track_no, rng, family, t0=1.7e9):
    """One simulated ICD track. Returns (DataFrame in ICD format, labels per packet, truth/meta dict)."""
    fam = FAMILIES[family]
    period = float(rng.choice(fam["periods"]))
    n_rep = int(rng.integers(30, 105))
    total = n_rep * period + 60
    mimic = MIMIC.get(cls) if rng.random() < P_DISGUISE else None
    breakout = float(rng.uniform(0.35, 0.65) * n_rep * period) if mimic else None
    t, Pt, Vt, beh = simulate_truth(cls, rng, total, mimic, breakout)
    t_rep = np.cumsum(np.r_[0.0, period * (1 + rng.normal(0, 0.012, n_rep - 1))])
    # missing packets (blackouts): the tracker emits nothing for a few periods
    keep = np.ones(n_rep, bool)
    for _ in range(rng.poisson(0.8)):
        s = int(rng.integers(3, max(4, n_rep - 4))); keep[s:s + int(rng.integers(1, 4))] = False
    t_rep = t_rep[keep]; n = len(t_rep)
    Pi = np.stack([np.interp(t_rep, t, Pt[:, a]) for a in range(3)], 1)
    Vi = np.stack([np.interp(t_rep, t, Vt[:, a]) for a in range(3)], 1)
    est, sd, cv, upd = kalman_track(t_rep, Pi, Vi, rng, fam)
    p, v, a = est[:, :, 0], est[:, :, 1], est[:, :, 2]
    # label = the class the target is *acting like* at that packet (changes after a disguise breakout)
    idx = np.clip(np.searchsorted(t, t_rep), 0, len(t) - 1)
    lab = np.array([beh[i] for i in idx], dtype=object)
    if mimic:
        lab = np.where(t_rep < breakout + 1.5 * period, mimic, cls)
    tt = t0 + t_rep
    df = pd.DataFrame({
        "packet_size": 166, "source": 7, "destination": 255, "packet_type": 163, "packet_category": 11,
        "radar_id": 1, "track_no": track_no,
        "pos_x": p[:, 0], "pos_y": p[:, 1], "pos_z": p[:, 2], "range_mag": np.linalg.norm(p, axis=1),
        "vel_x": v[:, 0], "vel_y": v[:, 1], "vel_z": v[:, 2], "vel_mag": np.linalg.norm(v, axis=1),
        "heading": np.degrees(np.arctan2(v[:, 1], v[:, 0])) % 360,
        "accel_x": a[:, 0], "accel_y": a[:, 1], "accel_z": a[:, 2], "accel_mag": np.linalg.norm(a, axis=1),
        "std_dev_x": sd[:, 0, 0], "std_dev_y": sd[:, 1, 0], "std_dev_z": sd[:, 2, 0],
        "std_dev_vx": sd[:, 0, 1], "std_dev_vy": sd[:, 1, 1], "std_dev_vz": sd[:, 2, 1],
        "std_dev_ax": sd[:, 0, 2], "std_dev_ay": sd[:, 1, 2], "std_dev_az": sd[:, 2, 2],
        "cov_x_vx": cv[:, 0], "cov_y_vy": cv[:, 1], "cov_z_vz": cv[:, 2],
        "ntp_sec": np.floor(tt).astype(np.int64), "ntp_frac": ((tt % 1) * 1e6).astype(np.int64),
        "assoc_indication": upd.astype(int)})
    meta = dict(cls=cls, mimic=mimic, breakout_idx=(int(np.argmax(t_rep >= breakout + 1.5 * period)) if mimic and (t_rep >= breakout + 1.5 * period).any() else -1),
                family=family, period=period)
    return df, lab, meta


def generate_dataset(n_per_class, families, rng, first_track=1000):
    frames, labels, metas = [], [], []
    tn = first_track
    for fam_name, count in zip(families, n_per_class):
        for cls in CLASSES:
            for _ in range(count):
                df, lab, meta = make_track(cls, tn, rng, fam_name)
                df["track_no"] = tn; frames.append(df); labels.append(lab); metas.append(meta); tn += 1
    return frames, labels, metas

if __name__ == "__main__":
    pass

## Cell 5 - Reading ICD files + the feature extractor
**One** function turns any ICD table (simulated or real, CSV or Excel) into the same clean packet table, and **one**
function turns a track into features. The same code runs in training and in live inference, so there is no mismatch.

Design rules learned from the failures of v11 / v12 / v19:
* **No absolute position, no `range_mag`, no `std_dev_*` / `cov_*` as classifier inputs.** Where the target happens to be
  relative to the radar, or how pessimistic the tracker's covariance is, says nothing about *what* the target is.
  (v11 learned "std ~ 1400 => SHIP" from its own simulator; the real tracker reports completely different values.)
* **No dependence on `ntp` spacing.** Your `crossing_2.xlsx` has timestamps 2.5 s and 11 s apart while the positions say
  ~10 s, so rate features built from timestamps would be wrong. We use the tracker's own `vel` and `accel` instead.
* Features are physical and scale-free: altitude, speed, vertical rate, lateral/longitudinal acceleration, turn rate,
  how much these vary, and the maxima seen so far.
* The network is trained on histories of **every length (1, 2, 3 ... packets)** and is told how long the history is.

In [ ]:
FEATURES = ["alt_log", "speed_log", "hspeed_log", "vz_ratio", "vz_asinh", "acc_log", "alat_log", "along_asinh",
            "az_asinh", "turn_log", "dhead_log", "r_speed_std", "r_vz_std", "r_alat_mean", "r_alat_max",
            "r_along_abs", "r_turn_mean", "cum_alat_max", "cum_acc_max", "cum_speed_max", "cum_alt_range",
            "cum_speed_range", "age_log", "coast", "coast_roll"]
NF = len(FEATURES)


def prepare_packets(raw, source="data"):
    """Normalise any ICD-style table into: track_key, t [s], pos/vel/acc in SI units, coast flag, _row."""
    df = raw.copy()
    df.columns = [str(c).strip() for c in df.columns]
    num = lambda c: pd.to_numeric(df[c], errors="coerce") if c in df.columns else pd.Series(np.nan, index=df.index)
    out = pd.DataFrame(index=df.index)
    out["_row"] = np.arange(len(df))
    out["track_key"] = df["track_no"].astype(str) if "track_no" in df.columns else "1"
    sec = num("ntp_sec"); frac = num("ntp_frac").fillna(0.0)
    out["t"] = (sec + frac * 1e-6) if sec.notna().any() else pd.Series(np.arange(len(df)) * 10.0, index=df.index)
    for a in "xyz":
        out[f"pos_{a}"] = num(f"pos_{a}") * (ALT_SCALE if a == "z" else 1.0)
        out[f"vel_{a}"] = num(f"vel_{a}") * VEL_SCALE
        out[f"acc_{a}"] = num(f"accel_{a}") * VEL_SCALE
    out["coast"] = (num("assoc_indication").fillna(1.0) == 0).astype(float) if "assoc_indication" in df.columns else 0.0
    for c in ["std_dev_x", "std_dev_y", "std_dev_z", "std_dev_vx", "std_dev_vy", "std_dev_vz", "meas_range"]:
        out[c] = num(c)                                 # kept for diagnostics / plots only
    out["source"] = source
    out = out.dropna(subset=["t", "pos_x", "pos_y", "pos_z"]).sort_values(["track_key", "t"], kind="stable")
    # derive missing velocity / acceleration from positions (only if the ICD column is absent)
    for g_key, g in out.groupby("track_key", sort=False):
        dt = g["t"].diff().clip(lower=0.5, upper=30.0)
        for a in "xyz":
            if g[f"vel_{a}"].isna().all():
                out.loc[g.index, f"vel_{a}"] = (g[f"pos_{a}"].diff() / dt).fillna(0.0)
        g = out.loc[g.index]
        for a in "xyz":
            if g[f"acc_{a}"].isna().all():
                out.loc[g.index, f"acc_{a}"] = (g[f"vel_{a}"].diff() / dt).fillna(0.0)
    for c in ["vel_x", "vel_y", "vel_z", "acc_x", "acc_y", "acc_z"]:
        out[c] = out[c].fillna(0.0)
    return out.reset_index(drop=True)


def _roll(x, w, fn):
    return getattr(pd.Series(x).rolling(w, min_periods=1), fn)().to_numpy()


def track_features(g):
    """g: packet table of ONE track (sorted). Returns features (n,NF) and physical series (n,4)=[alt,speed,|vz|,lat g]."""
    V = g[["vel_x", "vel_y", "vel_z"]].to_numpy(float); A = g[["acc_x", "acc_y", "acc_z"]].to_numpy(float)
    z = np.maximum(g["pos_z"].to_numpy(float), 0.0)
    n = len(g)
    speed = np.linalg.norm(V, axis=1); hs = np.hypot(V[:, 0], V[:, 1]); amag = np.linalg.norm(A, axis=1)
    vhat = V / np.maximum(speed, 1.0)[:, None]
    a_along = (A * vhat).sum(1)
    a_lat = np.linalg.norm(A - a_along[:, None] * vhat, axis=1)
    omega = np.abs(V[:, 0] * A[:, 1] - V[:, 1] * A[:, 0]) / (hs**2 + 4.0)          # horizontal turn rate [rad/s]
    hd = np.unwrap(np.arctan2(V[:, 1], V[:, 0]))
    dh = np.degrees(np.abs(np.diff(hd, prepend=hd[0]))) * (hs > 1.0)
    coast = g["coast"].to_numpy(float)
    L = lambda x, s=0.1: np.log1p(np.abs(x) / s)
    f = np.column_stack([
        np.log1p(z), np.log1p(speed), np.log1p(hs), np.clip(V[:, 2] / (speed + 1.0), -1, 1), np.arcsinh(V[:, 2] / 2.0),
        L(amag), L(a_lat), np.arcsinh(a_along / 0.1), np.arcsinh(A[:, 2] / 0.1), np.log1p(1000 * omega), np.log1p(dh),
        np.log1p(_roll(speed, 5, "std").clip(0)), np.log1p(_roll(V[:, 2], 5, "std").clip(0)),
        L(_roll(a_lat, 5, "mean")), L(_roll(a_lat, 5, "max")), L(_roll(np.abs(a_along), 5, "mean")),
        np.log1p(1000 * _roll(omega, 5, "mean")),
        L(np.maximum.accumulate(a_lat)), L(np.maximum.accumulate(amag)), np.log1p(np.maximum.accumulate(speed)),
        np.log1p(np.maximum.accumulate(z) - np.minimum.accumulate(z)),
        np.log1p(np.maximum.accumulate(speed) - np.minimum.accumulate(speed)),
        np.log1p(np.arange(1, n + 1)), coast, _roll(coast, 5, "mean")])
    f = np.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    phys = np.column_stack([z, speed, np.abs(V[:, 2]), a_lat / G0]).astype(np.float32)
    return f, phys


def build_track_table(frames, labels=None, metas=None):
    """frames: list of ICD DataFrames. Returns list of dicts, one per track."""
    tracks = []
    for i, raw in enumerate(frames):
        pk = prepare_packets(raw)
        for key, g in pk.groupby("track_key", sort=False):
            g = g.reset_index(drop=True)
            f, phys = track_features(g)
            tr = dict(key=key, packets=g, feats=f, phys=phys)
            if labels is not None:
                tr["labels"] = np.asarray(labels[i], dtype=object); tr["meta"] = metas[i]; tr["raw"] = raw
            tracks.append(tr)
    return tracks


# ---------------- physics feasibility gate (shared by training analysis and live inference) ----------------
def _soft_window(x, lo, hi, s):
    xl, lo_l, hi_l = np.log1p(x), np.log1p(lo), np.log1p(hi)
    d = np.maximum(lo_l - xl, 0.0) + np.maximum(xl - hi_l, 0.0)
    return np.exp(-(d / s) ** 2)


def feasibility(phys_hist, floor=1e-4):
    """phys_hist: (m,4) = last packets [alt, speed, |vz|, lateral g]. Returns (K,) in [floor,1]."""
    h = np.asarray(phys_hist, float)[-5:]
    alt, spd, vz, g = np.median(h[:, 0]), np.median(h[:, 1]), np.median(h[:, 2]), np.percentile(h[:, 3], 80)
    out = np.empty(K)
    for i, c in enumerate(CLASSES):
        e = ENVELOPE[c]
        out[i] = (_soft_window(alt, *e["alt"], 0.5) * _soft_window(spd, *e["speed"], 0.4)
                  * _soft_window(vz, 0, e["vz"], 0.8) * _soft_window(max(g - 0.02, 0.0), 0, e["g"], 0.8))
    return np.maximum(out, floor)
print("Feature extractor ready:", NF, "features")

## Cell 6 - Build the training / validation / test data
Every split uses *different tracks* and different "sensor weather" (see `FAMILIES`). Validation is deliberately
harder than training, and `test_stress` harder still, so the scores below are honest about robustness.
Windows are cut at **every** packet position, so the network sees histories of length 1, 2, 3 ... exactly like a live feed.

In [ ]:
t_start = time.time()
rng = np.random.default_rng(SEED)
_next_id = [1000]


def make_split(n_per_class, family_names):
    fr, lb, me = generate_dataset(n_per_class, family_names, rng, first_track=_next_id[0])
    _next_id[0] += sum(n_per_class) * K + 10
    return build_track_table(fr, lb, me)

_c = TRACKS_PER_CLASS
train_tracks = make_split([_c["train"] // 3, _c["train"] // 3, _c["train"] - 2 * (_c["train"] // 3)],
                          ["nominal_train", "noisy_train", "mixed_train"])
val_tracks = make_split([_c["val"]], ["val_shift"])
iid_tracks = make_split([_c["test_iid"]], ["test_iid"])
stress_tracks = make_split([_c["test_stress"]], ["test_stress"])
print(f"tracks  train {len(train_tracks)} | val {len(val_tracks)} | iid test {len(iid_tracks)} | stress test {len(stress_tracks)}"
      f"   ({time.time() - t_start:.0f}s)")
n_dis = sum(1 for t in train_tracks if t["meta"]["mimic"])
print(f"disguised training tracks: {n_dis} ({n_dis / len(train_tracks):.0%})")

# feature scaling is fitted on training packets only
_all = np.vstack([t["feats"] for t in train_tracks])
FMEAN = _all.mean(0).astype(np.float32); FSTD = np.maximum(_all.std(0), 1e-3).astype(np.float32)


def scale(f):
    return np.clip((f - FMEAN) / FSTD, -6, 6).astype(np.float32)


def make_windows(tracks, stride_late=3, early_n=12):
    Xs, ys, meta = [], [], []
    for ti, tr in enumerate(tracks):
        fs = scale(tr["feats"]); n = len(fs)
        lab = np.array([CLS_ID[c] for c in tr["labels"]])
        for k in range(n):
            if not (k < early_n or (k - early_n) % stride_late == 0 or k == n - 1):
                continue
            lo = max(0, k - SEQ_LEN + 1); w = fs[lo:k + 1]; pad = SEQ_LEN - len(w)
            arr = np.zeros((SEQ_LEN, NF + 1), np.float32); arr[pad:, :NF] = w; arr[pad:, NF] = 1.0
            Xs.append(arr); ys.append(lab[k])
            meta.append((ti, k, k + 1, tr["meta"]["breakout_idx"] if tr["meta"]["mimic"] else -1))
    return np.stack(Xs), np.array(ys), pd.DataFrame(meta, columns=["track", "k", "n_hist", "breakout"])

X_tr, y_tr, m_tr = make_windows(train_tracks)
X_va, y_va, m_va = make_windows(val_tracks, stride_late=1)
X_iid, y_iid, m_iid = make_windows(iid_tracks, stride_late=1)
X_st, y_st, m_st = make_windows(stress_tracks, stride_late=1)
print("windows:", X_tr.shape, X_va.shape, X_iid.shape, X_st.shape)

## Cell 7 - The network (+ training, + probability calibration)
Same family as before (Conv -> BiGRU -> attention), but: it outputs **logits**, it sees a *valid-step mask* channel,
and after training a single **temperature** is fitted on the (harder) validation set so that "80 %" really means
about 80 % - the old models printed 97 % on 1-3 packets of history.

In [ ]:
def build_model():
    inp = layers.Input((SEQ_LEN, NF + 1))
    x0 = layers.GaussianNoise(0.15)(inp)                     # light input noise = regulariser (training only)
    c1 = layers.Conv1D(32, 3, padding="same", activation="relu")(x0)
    c2 = layers.Conv1D(32, 5, padding="same", activation="relu")(x0)
    x = layers.BatchNormalization()(layers.Concatenate()([c1, c2]))
    x = layers.Bidirectional(layers.GRU(32, return_sequences=True, dropout=0.2))(x)
    att = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=0.15)(x, x)
    x = layers.LayerNormalization()(x + att)
    last = x[:, -1, :]                                       # newest packet matters most in a live feed
    pooled = layers.Concatenate()([layers.GlobalAveragePooling1D()(x), layers.GlobalMaxPooling1D()(x), last])
    h = layers.Dense(64, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(pooled)
    h = layers.Dropout(0.4)(layers.BatchNormalization()(h))
    out = layers.Dense(K)(h)                                 # logits
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(2e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
    return m

model = build_model()
cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
       keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5)]
t_start = time.time()
history = model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                    callbacks=cbs, verbose=2)
print(f"training time {time.time() - t_start:.0f}s")

logits_va = model.predict(X_va, batch_size=1024, verbose=0)
def _nll(T): return -np.mean(np.log(softmax(logits_va / T, axis=1)[np.arange(len(y_va)), y_va] + 1e-12))
TEMP = float(minimize_scalar(_nll, bounds=(0.3, 6.0), method="bounded").x)
print(f"calibration temperature T = {TEMP:.2f}   (NLL {_nll(1.0):.3f} -> {_nll(TEMP):.3f})")

## Cell 8 - Temporal fusion: from per-packet guesses to a stable, switchable belief
A raw network output jumps around from packet to packet. A live display needs a **belief** per track that
(1) accumulates evidence, (2) can still **change its mind** when the target changes behaviour (disguise breakout),
and (3) is multiplied by the **physics gate** so impossible classes are suppressed.

This is a small Bayesian filter: `belief_t  ~  [(1-eps) * belief_(t-1) + eps/(K-1) * (others)]  x  (p_net x feasibility)^w`.
`eps` = how easily the target may switch class, `w` = how much a single packet is trusted (smaller when the radar
coasted). Both are **tuned automatically on the validation tracks** just below.

In [ ]:
GROUP_OF = {"CIVILIAN_AIRCRAFT": "AIRCRAFT", "FIGHTER_AIRCRAFT": "AIRCRAFT", "HELICOPTER": "ROTARY/UAV", "DRONE": "ROTARY/UAV",
            "SURFACE_MISSILE": "MISSILE", "AERIAL_MISSILE": "MISSILE", "SKIFF": "VESSEL", "BOAT": "VESSEL", "SHIP": "VESSEL"}
GROUP_ID = np.array([sorted(set(GROUP_OF.values())).index(GROUP_OF[c]) for c in CLASSES])
DOM_ID = np.array([0 if DOMAIN_OF[c] == "AIR" else 1 for c in CLASSES])
BELIEF = dict(eps=0.03, w=0.6, coast_w=0.5, gamma=1.0)


def belief_step(b_prev, p, feas, coast, prm):
    e = p * feas ** prm["gamma"]; e = np.maximum(e / e.sum(), 1e-9)
    w = prm["w"] * (prm["coast_w"] if coast else 1.0)
    pred = (1 - prm["eps"]) * b_prev + prm["eps"] / (K - 1) * (1 - b_prev)
    b = pred * e ** w
    return b / b.sum()


def window_probs(model, X, batch=2048):
    return softmax(model.predict(X, batch_size=batch, verbose=0) / TEMP, axis=1)


def per_track_sequences(tracks, X, m, probs):
    """Re-assemble window outputs into one (n,K) array per track (windows were cut at every packet)."""
    seqs = [None] * len(tracks)
    for ti, g in m.groupby("track"):
        order = g.sort_values("k"); seqs[ti] = probs[order.index.to_numpy()]
    return seqs


def feas_sequence(track):
    """Feasibility of every class at every packet (cached - it only depends on the physics, not on the network)."""
    if "feas" not in track:
        ph = track["phys"]; track["feas"] = np.stack([feasibility(ph[max(0, k - 4):k + 1]) for k in range(len(ph))])
    return track["feas"]


def run_belief(track, p_seq, prm, use_gate=True):
    n = len(p_seq); B = np.zeros((n, K)); b = np.ones(K) / K
    coast = track["packets"]["coast"].to_numpy(); FE = feas_sequence(track) if use_gate else np.ones((n, K))
    for k in range(n):
        b = belief_step(b, p_seq[k], FE[k], coast[k] > 0.5, prm); B[k] = b
    return B


def label_ids(track):
    return np.array([CLS_ID[c] for c in track["labels"]])


def eval_fusion(tracks, seqs, prm, use_gate=True):
    ok = dict(dom=[], grp=[], cls=[]); post = []
    for tr, ps in zip(tracks, seqs):
        B = run_belief(tr, ps, prm, use_gate); pred = B.argmax(1); y = label_ids(tr)
        ok["cls"].append(pred == y); ok["grp"].append(GROUP_ID[pred] == GROUP_ID[y]); ok["dom"].append(DOM_ID[pred] == DOM_ID[y])
        bo = tr["meta"]["breakout_idx"]
        if tr["meta"]["mimic"] and bo >= 0: post.append((pred == y)[bo:])
    r = {k: float(np.mean(np.concatenate(v))) for k, v in ok.items()}
    r["post_breakout"] = float(np.mean(np.concatenate(post))) if post else np.nan
    return r

def flips_per_100(tracks, seqs, prm=None):
    ch = tot = 0
    for tr, ps in zip(tracks, seqs):
        pred = ps.argmax(1) if prm is None else run_belief(tr, ps, prm).argmax(1)
        ch += int((pred[1:] != pred[:-1]).sum()); tot += len(pred) - 1
    return 100.0 * ch / max(tot, 1)

p_va = window_probs(model, X_va)
seq_va = per_track_sequences(val_tracks, X_va, m_va, p_va)
grid = [(e, w) for e in (0.01, 0.05, 0.15, 0.3) for w in (0.3, 0.6, 1.0, 1.5)]
rows = []
for e, w in grid:
    r = eval_fusion(val_tracks, seq_va, dict(BELIEF, eps=e, w=w)); rows.append(dict(eps=e, w=w, **r))
tune = pd.DataFrame(rows)
tune["score"] = 0.5 * tune["cls"] + 0.5 * tune["post_breakout"]       # accuracy AND ability to change its mind
tune = tune.sort_values("score", ascending=False)
best = tune.iloc[0]; BELIEF.update(eps=float(best.eps), w=float(best.w))
print("fusion tuning on validation tracks:"); display(tune.head(6).round(3))
print("chosen:", {k: BELIEF[k] for k in ("eps", "w")})

# the belief recursion over-counts overlapping windows, so a 2nd temperature makes the *displayed* confidence honest
_B = np.vstack([run_belief(t, ps, BELIEF) for t, ps in zip(val_tracks, seq_va)]); _y = np.concatenate([label_ids(t) for t in val_tracks])
def _nllb(T): return -np.mean(np.log(softmax(np.log(np.maximum(_B, 1e-9)) / T, axis=1)[np.arange(len(_y)), _y] + 1e-12))
TEMP_B = float(minimize_scalar(_nllb, bounds=(0.5, 12.0), method="bounded").x)
def calib(B):  # belief -> calibrated confidence (argmax unchanged)
    return softmax(np.log(np.maximum(B, 1e-9)) / TEMP_B, axis=-1)
print(f"belief temperature = {TEMP_B:.2f}  (NLL {_nllb(1.0):.3f} -> {_nllb(TEMP_B):.3f})")

## Cell 9 - How good is it? (held-out simulated tracks, 3 tiers of difficulty)
Tier 1 = **domain** (AIR vs SURFACE) - what you complained about. Tier 2 = **group** (aircraft / rotary-UAV / missile / vessel).
Tier 3 = **exact class** (9 classes). Classes such as SKIFF / BOAT / SHIP or HELICOPTER / DRONE overlap physically, so tier 3 is
naturally lower - the engine then reports the top-2 instead of pretending to be sure.

In [ ]:
def tier_table(tracks, X, m, name):
    raw_p = window_probs(model, X); seqs = per_track_sequences(tracks, X, m, raw_p)
    y = np.concatenate([label_ids(t) for t in tracks]); pr = np.concatenate([s.argmax(1) for s in seqs])
    raw = dict(domain=np.mean(DOM_ID[pr] == DOM_ID[y]), group=np.mean(GROUP_ID[pr] == GROUP_ID[y]), cls=np.mean(pr == y))
    g = eval_fusion(tracks, seqs, dict(BELIEF, eps=0.5, w=1.0)); f = eval_fusion(tracks, seqs, BELIEF)   # gate only / full
    rows = {"raw network": raw, "+ physics gate": dict(domain=g["dom"], group=g["grp"], cls=g["cls"]),
            "+ fusion (final)": dict(domain=f["dom"], group=f["grp"], cls=f["cls"])}
    out = pd.DataFrame(rows).T.rename(columns={"cls": "class"}); out.insert(0, "split", name)
    out["label flips /100 pkts"] = [flips_per_100(tracks, seqs), flips_per_100(tracks, seqs, dict(BELIEF, eps=0.5, w=1.0)), flips_per_100(tracks, seqs, BELIEF)]
    return out, seqs

tab_iid, seq_iid = tier_table(iid_tracks, X_iid, m_iid, "unseen tracks, familiar sensor")
tab_st, seq_st = tier_table(stress_tracks, X_st, m_st, "unseen tracks, stress sensor")
display(pd.concat([tab_iid, tab_st]).round(3))

# track-level result = verdict after the LAST packet of each track vs the true (post-breakout) class
def final_verdicts(tracks, seqs):
    out = []
    for tr, ps in zip(tracks, seqs):
        B = run_belief(tr, ps, BELIEF); out.append((tr["labels"][-1], CLASSES[int(B[-1].argmax())], float(B[-1].max()), tr["meta"]["mimic"] is not None))
    return pd.DataFrame(out, columns=["true", "pred", "conf", "disguised"])
fv_iid, fv_st = final_verdicts(iid_tracks, seq_iid), final_verdicts(stress_tracks, seq_st)
print(f"track-level final-verdict accuracy: iid {np.mean(fv_iid.true == fv_iid.pred):.1%} | stress {np.mean(fv_st.true == fv_st.pred):.1%}")

## Cell 10 - Evaluation plots (what to show your manager)
1. Confusion matrices of the final verdict per track. 2. Accuracy vs. packets received (how fast it locks on).
3. Calibration (is "80 %" really 80 %?). 4. Disguise breakout: how many packets until the belief flips to the true class.
5. Training curves.

In [ ]:
def plot_confusion(ax, fv, title):
    cm = confusion_matrix(fv.true, fv.pred, labels=CLASSES); sup = cm.sum(1)
    pct = np.divide(cm, sup[:, None], out=np.zeros(cm.shape), where=sup[:, None] > 0) * 100
    ax.imshow(pct, cmap="Blues", vmin=0, vmax=100)
    short = [c.replace("_AIRCRAFT", "").replace("SURFACE_MISSILE", "SURF.MSL").replace("AERIAL_MISSILE", "AIR.MSL").replace("HELICOPTER", "HELI") for c in CLASSES]
    ax.set_xticks(range(K), short, rotation=45, ha="right", fontsize=8); ax.set_yticks(range(K), short, fontsize=8)
    for i in range(K):
        for j in range(K):
            if cm[i, j]: ax.text(j, i, f"{pct[i, j]:.0f}", ha="center", va="center", fontsize=8, color="white" if pct[i, j] > 55 else "black")
    ax.set_title(f"{title}\ntrack accuracy {np.mean(fv.true == fv.pred):.1%}", fontsize=10); ax.set_xlabel("predicted"); ax.set_ylabel("true")

fig, axs = plt.subplots(2, 3, figsize=(17, 9.5))
plot_confusion(axs[0, 0], fv_iid, "Final verdict - familiar sensor"); plot_confusion(axs[0, 1], fv_st, "Final verdict - stress sensor")

# accuracy vs history length (packet-level, fused)
def acc_by_n(tracks, seqs):
    xs = np.arange(1, 61); hit = np.zeros(60); cnt = np.zeros(60); hit_r = np.zeros(60)
    for tr, ps in zip(tracks, seqs):
        B = run_belief(tr, ps, BELIEF); y = label_ids(tr); L = min(len(y), 60)
        hit[:L] += (B.argmax(1) == y)[:L]; hit_r[:L] += (ps.argmax(1) == y)[:L]; cnt[:L] += 1
    return xs, hit / np.maximum(cnt, 1), hit_r / np.maximum(cnt, 1), cnt
for nm, tr_, sq_, c in [("familiar", iid_tracks, seq_iid, "#1f77b4"), ("stress", stress_tracks, seq_st, "#d62728")]:
    xs, a_f, a_r, cnt = acc_by_n(tr_, sq_); ok = cnt > 20
    axs[0, 2].plot(xs[ok], a_f[ok], color=c, lw=2, label=f"{nm}: fused"); axs[0, 2].plot(xs[ok], a_r[ok], color=c, lw=1, ls="--", label=f"{nm}: raw network")
axs[0, 2].set_xlabel("packets received on the track"); axs[0, 2].set_ylabel("class accuracy"); axs[0, 2].set_ylim(0, 1.02)
axs[0, 2].grid(alpha=.3); axs[0, 2].legend(fontsize=8); axs[0, 2].set_title("Class accuracy vs. packets received")

# calibration
conf, hit = [], []
for tr, ps in zip(iid_tracks + stress_tracks, seq_iid + seq_st):
    B = calib(run_belief(tr, ps, BELIEF)); conf.append(B.max(1)); hit.append(B.argmax(1) == label_ids(tr))
conf, hit = np.concatenate(conf), np.concatenate(hit)
edges = np.linspace(0.1, 1.0, 10); idx = np.digitize(conf, edges) - 1
bx, by, bn = [], [], []
for b in range(9):
    s = idx == b
    if s.sum() > 30: bx.append(conf[s].mean()); by.append(hit[s].mean()); bn.append(s.sum())
ECE = float(np.sum(np.abs(np.array(bx) - np.array(by)) * np.array(bn)) / np.sum(bn))
axs[1, 0].plot([0, 1], [0, 1], "k--", lw=1); axs[1, 0].plot(bx, by, "o-", color="#2ca02c")
axs[1, 0].set_xlabel("reported confidence"); axs[1, 0].set_ylabel("observed accuracy"); axs[1, 0].grid(alpha=.3)
axs[1, 0].set_title(f"Calibration (ECE {ECE:.3f}) - on the diagonal = honest")

# disguise breakout: (a) packets until verdict equals the TRUE class, (b) packets until the verdict changes at all
lat_true, lat_any = [], []
for tr, ps in zip(iid_tracks + stress_tracks, seq_iid + seq_st):
    bo = tr["meta"]["breakout_idx"]
    if tr["meta"]["mimic"] and bo >= 2:
        B = run_belief(tr, ps, BELIEF).argmax(1); y = label_ids(tr)
        h = np.where(B[bo:] == y[bo])[0]; lat_true.append(int(h[0]) if len(h) else 99)
        c = np.where(B[bo:] != B[bo - 1])[0]; lat_any.append(int(c[0]) if len(c) else 99)
lat_true, lat_any = np.array(lat_true), np.array(lat_any)
bins = np.arange(-0.5, 21.5, 1)
axs[1, 1].hist([np.minimum(lat_any, 20), np.minimum(lat_true, 20)], bins=bins, label=["verdict changes at all", "verdict = true class"], color=["#ff7f0e", "#2ca02c"])
axs[1, 1].set_xlabel("packets after the breakout (20 = not within 20)"); axs[1, 1].set_ylabel("disguised tracks"); axs[1, 1].legend(fontsize=8)
axs[1, 1].set_title(f"Disguise breakout: verdict changes within 5 pkts for {np.mean(lat_any <= 5):.0%}")
axs[1, 2].plot(history.history["loss"], label="train loss"); axs[1, 2].plot(history.history["val_loss"], label="validation loss")
axs[1, 2].set_xlabel("epoch"); axs[1, 2].legend(); axs[1, 2].grid(alpha=.3); axs[1, 2].set_title("Training curves (early stopping)")
plt.tight_layout(); plt.show()

## Cell 11 - The live engine (one object, one call per incoming packet)
`LiveClassifier.update(packet)` is what you would call from a socket / queue in deployment. It keeps a separate
buffer and belief **per track_no**, so interleaved tracks (like in `crossing_2.xlsx`) never mix.
Per packet it: builds features -> network -> calibrated probabilities -> x physics gate -> belief update -> verdict.

Verdict wording is deliberately honest:
* **PROVISIONAL** (< 4 packets): the domain (AIR / SURFACE) is usually already certain, the class is only a lean.
* **AMBIGUOUS**: two classes are close; both are shown (e.g. `BOAT or SKIFF`).
* **LIKELY / CONFIRMED**: enough history and enough margin.
* `+OOD`: the track does not look like anything in the simulator (trust the physics gate, not the class).

In [ ]:
# Out-of-distribution detector: how unusual is a feature vector compared with the simulated training world?
_Z = np.vstack([scale(t["feats"]) for t in train_tracks]); OOD_MU = _Z.mean(0)
OOD_INV = np.linalg.inv(np.cov(_Z.T) + 1e-2 * np.eye(NF))
def ood_d2(z): d = z - OOD_MU; return float(d @ OOD_INV @ d)
OOD_THR = float(np.percentile([ood_d2(z) for z in _Z[::7]], 99.7))
print(f"OOD threshold (99.7th percentile of the simulated world): {OOD_THR:.1f}")

_fwd = tf.function(lambda x: model(x, training=False))
MIN_PKTS, CONF_OK, CONF_MIN, MARGIN_MIN = 4, 0.80, 0.45, 0.10


class LiveClassifier:
    def __init__(self, buffer=160, prm=None):
        self.buffer = buffer; self.prm = prm or BELIEF; self.tracks = {}

    def _new(self):
        return dict(raw=[], b=np.ones(K) / K, stable=None, cand=None, streak=0, n=0, events=[])

    def update(self, row):
        key = str(row.get("track_no", "1")); st = self.tracks.setdefault(key, self._new())
        st["raw"].append(row); st["raw"] = st["raw"][-self.buffer:]; st["n"] += 1
        g = prepare_packets(pd.DataFrame(st["raw"]), source=key)
        g = g[g.track_key == key] if (g.track_key == key).any() else g
        f, phys = track_features(g); n = len(g)
        fs = scale(f[-SEQ_LEN:]); arr = np.zeros((1, SEQ_LEN, NF + 1), np.float32)
        arr[0, SEQ_LEN - len(fs):, :NF] = fs; arr[0, SEQ_LEN - len(fs):, NF] = 1.0
        p = softmax(_fwd(arr).numpy()[0] / TEMP)
        fe = feasibility(phys[-5:]); coast = bool(g["coast"].iloc[-1] > 0.5)
        st["b"] = belief_step(st["b"], p, fe, coast, self.prm)
        cb = calib(st["b"]); order = np.argsort(cb)[::-1]; top, sec = int(order[0]), int(order[1])
        conf, margin = float(cb[top]), float(cb[top] - cb[sec])
        p_air = float(cb[AIR_IDX].sum()); domain = "AIR" if p_air >= 0.5 else "SURFACE"
        ood = ood_d2(fs[-1]) > OOD_THR
        if n < MIN_PKTS: status = "PROVISIONAL"
        elif conf < CONF_MIN or margin < MARGIN_MIN: status = "AMBIGUOUS"
        elif conf >= CONF_OK and n >= 8: status = "CONFIRMED"
        else: status = "LIKELY"
        if status == "PROVISIONAL": verdict = f"{domain} (class pending, leaning {CLASSES[top]})"
        elif status == "AMBIGUOUS": verdict = f"{domain}: {CLASSES[top]} or {CLASSES[sec]}"
        else: verdict = CLASSES[top]
        # stable label with hysteresis (2 consecutive packets AND >=50 % to switch) -> this drives the class-change events
        event = None
        if st["stable"] is None:
            if conf >= 0.5 and n >= 2: st["stable"] = top
        elif top != st["stable"]:
            st["streak"] = st["streak"] + 1 if top == st["cand"] else 1; st["cand"] = top
            if st["streak"] >= 2 and conf >= 0.5:
                a, b2 = max(0, n - 7), max(1, n - 3)
                bef, aft = phys[a:b2], phys[b2:]
                trig = (f"alt {np.median(bef[:, 0]):.0f}->{np.median(aft[:, 0]):.0f} m, speed {np.median(bef[:, 1]):.0f}->{np.median(aft[:, 1]):.0f} m/s, "
                        f"lateral {np.median(bef[:, 3]):.2f}->{np.median(aft[:, 3]):.2f} g")
                event = dict(pkt=n, t=float(g["t"].iloc[-1]), frm=CLASSES[st["stable"]], to=CLASSES[top], conf=conf, trigger=trig)
                st["events"].append(event); st["stable"] = top; st["streak"] = 0
        else:
            st["streak"] = 0
        blocked = [CLASSES[i] for i in range(K) if p[i] > 0.15 and fe[i] < 0.05]
        a_g = float(phys[-1, 3])
        return dict(track=key, n=n, t=float(g["t"].iloc[-1]), alt=float(phys[-1, 0]), speed=float(phys[-1, 1]), vz=float(g["vel_z"].iloc[-1]),
                    lat_g=a_g, coast=coast, domain=domain, p_air=p_air, top=CLASSES[top], conf=conf, second=CLASSES[sec], conf2=float(cb[sec]),
                    margin=margin, status=status, verdict=verdict, stable=CLASSES[st["stable"]] if st["stable"] is not None else "UNDETERMINED",
                    ood=bool(ood), gate_blocked=",".join(blocked), event=event, cb=cb, p_raw=p, feas=fe,
                    x=float(g["pos_x"].iloc[-1]), y=float(g["pos_y"].iloc[-1]),
                    sig_pos=float(np.nanmean([g["std_dev_x"].iloc[-1], g["std_dev_y"].iloc[-1]])) if "std_dev_x" in g else np.nan)

print("LiveClassifier ready")

## Cell 12 - Load your radar file(s) + data audit
Accepts `.csv`, `.xlsx`, `.xls`. One or many files. If none of the names in `INPUT_FILES` is found, Colab shows an upload
button. The **audit** table checks things that silently break classifiers:
* is the timestamp spacing consistent with position/velocity? (`crossing_2.xlsx`: 2.5 s and 11 s gaps, but the positions imply ~10 s)
* does the velocity point the same way the position moves? (`cos` near +1 = yes). In **both** your files vel X/Y behave as if swapped
  relative to pos X/Y (`cos_if_xy_swapped` ~ +1). The classifier only uses magnitudes (speed, |a|, turn rate), which are unaffected.
We **do not** auto-correct anything; we only warn. The classifier uses neither timestamps nor vector directions, so it is unaffected by both.

In [ ]:
def find_file(name):
    cands = [Path(name), Path("/content") / name, Path("/mnt/user-data/uploads") / name, Path.home() / name]
    cands += list(Path("/mnt/user-data/attachments").glob(f"*/*{Path(name).name}")) if Path("/mnt/user-data/attachments").exists() else []
    return next((p for p in cands if p.exists()), None)


def read_any(path):
    if str(path).lower().endswith((".xlsx", ".xls", ".xlsm")):
        return pd.read_excel(path)
    return pd.read_csv(path, sep=None, engine="python")

found = [(n, find_file(n)) for n in INPUT_FILES]
if not any(p for _, p in found):
    try:
        from google.colab import files as _cf
        up = _cf.upload(); found = [(k, Path(k)) for k in up.keys()]
    except Exception:
        found = [(p, Path(p)) for p in input("Path(s) to radar file(s), comma separated: ").split(",")]
frames_real = []
for name, path in found:
    if path is None:
        print(f"  !! not found: {name}"); continue
    d = read_any(path); d = d.dropna(how="all")
    import re; stem = re.sub(r"^[0-9a-f]{8}-", "", Path(path).stem)       # drop upload-hash prefixes
    d["track_no"] = d["track_no"].astype(str).str.replace(r"\.0$", "", regex=True) if "track_no" in d.columns else "1"
    if len([1 for _, p in found if p is not None]) > 1:
        d["track_no"] = stem + "#" + d["track_no"]
    d["_file"] = stem; frames_real.append(d); print(f"loaded {path}: {len(d)} packets, tracks {sorted(d.track_no.unique())}")
raw_real = pd.concat(frames_real, ignore_index=True)
real_pk = prepare_packets(raw_real)


def audit_tracks(pk):
    rows = []
    for key, g in pk.groupby("track_key", sort=False):
        g = g.sort_values("t")
        n = len(g)
        if n < 2:
            rows.append(dict(track=key, packets=n, note="single packet - no audit possible")); continue
        dt = np.diff(g["t"].to_numpy()); dxy = np.diff(g[["pos_x", "pos_y"]].to_numpy(), axis=0)
        vxy = g[["vel_x", "vel_y"]].to_numpy(); vm = 0.5 * (vxy[1:] + vxy[:-1]); sp = np.linalg.norm(vm, axis=1)
        dt_kin = np.where(sp > 5, np.linalg.norm(dxy, axis=1) / np.maximum(sp, 1e-6), np.nan)
        nrm = np.linalg.norm(dxy, axis=1) * np.maximum(sp, 1e-6) + 1e-9
        cos = np.sum(dxy * vm, axis=1) / nrm
        vsw = vm[:, ::-1]; cos_sw = np.sum(dxy * vsw, axis=1) / nrm                  # same test with vel x<->y swapped
        ratio = np.nanmedian(dt / dt_kin) if np.isfinite(dt_kin).any() else np.nan
        notes = []
        if np.isfinite(ratio) and not (0.7 <= ratio <= 1.4): notes.append("TIMESTAMPS inconsistent with motion (classifier does not use them)")
        if np.nanmedian(cos) < 0.5 and np.nanmedian(cos_sw) > 0.8:
            notes.append("vel/accel X<->Y look SWAPPED vs pos (heading = atan2(vel_y,vel_x) is a bearing) - classifier unaffected, map is fine")
        elif np.nanmedian(cos) < 0.5:
            notes.append("VELOCITY does not follow position change - check axes/sign")
        rows.append(dict(track=key, packets=n, median_dt_ntp_s=np.median(dt), median_dt_from_motion_s=np.nanmedian(dt_kin),
                         ntp_over_motion=ratio, cos_vel_vs_dpos=np.nanmedian(cos), cos_if_xy_swapped=np.nanmedian(cos_sw), alt_m=g.pos_z.median(), speed_mps=np.linalg.norm(g[["vel_x", "vel_y", "vel_z"]], axis=1).mean(),
                         coast_frac=g.coast.mean(), note="; ".join(notes) if notes else "ok"))
    return pd.DataFrame(rows)

audit = audit_tracks(real_pk); display(audit.round(2))

## Cell 13 - LIVE REPLAY of the file(s)
Packets are fed in timestamp order, exactly as a live feed would deliver them. Each line = one packet:
`domain | verdict | confidence | status | kinematics`. A `>> CHANGE` line appears when the (hysteresis-protected) class changes.
`PRINT_MODE = "changes"` prints only status/class changes (use it for long files). `PLAYBACK_DELAY` slows it down for a demo.

In [ ]:
def replay(raw, clf, delay=PLAYBACK_DELAY, mode=PRINT_MODE, verbose=True):
    tt = pd.to_numeric(raw.get("ntp_sec"), errors="coerce").fillna(0) + pd.to_numeric(raw.get("ntp_frac", 0), errors="coerce").fillna(0) * 1e-6
    ordered = raw.assign(_t=tt.to_numpy()).sort_values("_t", kind="stable").reset_index(drop=True)
    totals = ordered.groupby("track_no").size().to_dict(); t0 = {k: v for k, v in ordered.groupby("track_no")["_t"].first().items()}
    rows, last = [], {}
    for i, rec in enumerate(ordered.drop(columns=["_t"]).to_dict("records")):
        r = clf.update(rec); r["g"] = i + 1; r["t_rel"] = r["t"] - t0[r["track"]]; rows.append(r)
        sig = (r["verdict"], r["status"])
        show = (mode == "every") or (last.get(r["track"]) != sig) or r["event"] is not None or r["n"] == totals[r["track"]]
        last[r["track"]] = sig
        if verbose and show:
            flag = " +OOD" if r["ood"] else ""
            print(f"[{r['g']:>4}/{len(ordered)}] trk {r['track']:<10} pkt {r['n']:>3} t=+{r['t_rel']:7.1f}s | {r['domain']:<7} | {r['verdict']:<46} "
                  f"{r['conf']:5.0%} | {r['status']:<11}{flag} | alt {r['alt']:7.0f} m  v {r['speed']:6.1f} m/s  lat {r['lat_g']:5.2f} g"
                  + (f"  [gate blocked: {r['gate_blocked']}]" if r["gate_blocked"] and r["n"] <= 3 else ""), flush=True)
            if r["event"]:
                e = r["event"]; print(f"      >> CHANGE on track {r['track']}: {e['frm']} -> {e['to']} at packet {e['pkt']} ({e['conf']:.0%}) | {e['trigger']}", flush=True)
            if delay: time.sleep(delay)
    return pd.DataFrame(rows)

clf = LiveClassifier()
stream = replay(raw_real, clf)
print(f"\nprocessed {len(stream)} packets on {stream.track.nunique()} track(s)")

## Cell 14 - Final summary per track (the "what is this target" report)
For each track: the verdict, how sure we are, the runner-up, the class timeline (with change events and *why* the class
changed), motion description, and data-quality flags. The table is saved to `radar_track_summary.csv`, the packet-by-packet
log to `radar_packet_log.csv`.

In [ ]:
def segments(labels):
    out, start = [], 0
    for i in range(1, len(labels) + 1):
        if i == len(labels) or labels[i] != labels[start]:
            out.append((start, i - 1, labels[start])); start = i
    return out


def motion_text(S):
    g = S["lat_g"].to_numpy(); v = S["speed"].to_numpy(); fr = lambda m: np.mean(m)
    parts = []
    if fr(v < 3) > 0.6: parts.append("mostly stationary / hovering")
    else:
        parts.append(f"cruising at {np.median(v):.0f} m/s" if np.ptp(v) < 0.25 * max(np.median(v), 1) else f"speed varies {v.min():.0f}-{v.max():.0f} m/s")
    if fr(g > 1.0) > 0.1: parts.append(f"hard manoeuvres up to {g.max():.1f} g")
    elif fr(g > 0.2) > 0.15: parts.append(f"sustained turning ({fr(g > 0.2):.0%} of the time, up to {g.max():.2f} g)")
    elif fr(g < 0.05) > 0.8: parts.append("flying straight")
    else: parts.append(f"gentle turns (max {g.max():.2f} g)")
    return "; ".join(parts)


def summarize(stream, audit_df=None):
    rows, texts = [], []
    for key, S in stream.groupby("track", sort=False):
        S = S.sort_values("n").reset_index(drop=True); n = len(S); B = np.stack(S.cb.values); last = S.iloc[-1]
        order = np.argsort(B[-1])[::-1]; top3 = [(CLASSES[i], float(B[-1][i])) for i in order[:3]]
        seg = segments(S["stable"].tolist()); share = S["stable"].value_counts(normalize=True)
        events = [e for e in S["events"].tolist() if e] if "events" in S else []
        dominant = CLASSES[int(B.mean(0).argmax())]
        dur = (S["t"].iloc[-1] - S["t"].iloc[0]) / 60
        note = ""
        if audit_df is not None and (audit_df.track == key).any(): note = audit_df[audit_df.track == key].note.iloc[0]
        rows.append(dict(track=key, packets=n, minutes=round(dur, 1), domain=last.domain, p_air=round(float(B[-1][AIR_IDX].sum()), 3),
                         verdict=last.verdict, final_class=last.top, confidence=round(last.conf, 3), status=last.status,
                         runner_up=top3[1][0], runner_up_conf=round(top3[1][1], 3), third=top3[2][0], dominant_over_time=dominant,
                         class_changes=len(events), alt_median_m=round(S.alt.median()), speed_median_mps=round(S.speed.median(), 1),
                         max_lateral_g=round(S.lat_g.max(), 2), coast_frac=round(S.coast.mean(), 2), ood_frac=round(S.ood.mean(), 2), audit=note))
        t = [f"TRACK {key}  -  {n} packets, {dur:.1f} min",
             f"  VERDICT      : {last.domain} -> {last.verdict}   [{last.status}, confidence {last.conf:.0%}]",
             f"  candidates   : " + " | ".join(f"{c} {p:.0%}" for c, p in top3),
             f"  domain       : {last.domain}  (P[air] = {B[-1][AIR_IDX].sum():.1%})",
             f"  class timeline: " + " -> ".join(f"{c} (pkt {a + 1}-{b + 1})" for a, b, c in seg),
             f"  motion       : alt {S.alt.median():.0f} m (range {S.alt.min():.0f}-{S.alt.max():.0f}); {motion_text(S)}"]
        for e in events: t.append(f"  change       : packet {e['pkt']}: {e['frm']} -> {e['to']} ({e['conf']:.0%})  [{e['trigger']}]")
        if n < MIN_PKTS * 2: t.append(f"  caution      : only {n} packets - the domain is reliable, the exact class is a lean (see candidates)")
        if S.ood.mean() > 0.3: t.append("  caution      : motion pattern is unlike the simulator (OOD) - rely on domain / physics, not on the class")
        if note and note != "ok": t.append(f"  data audit   : {note}")
        texts.append("\n".join(t))
    return pd.DataFrame(rows), texts

stream["events"] = stream["event"]
summary, texts = summarize(stream, audit)
print("=" * 100); print("  MISSION SUMMARY"); print("=" * 100)
for t in texts: print(t); print("-" * 100)
display(summary.drop(columns=["audit"]))
summary.to_csv("radar_track_summary.csv", index=False)
stream.drop(columns=["cb", "p_raw", "feas", "event", "events"]).to_csv("radar_packet_log.csv", index=False)
print("saved radar_track_summary.csv and radar_packet_log.csv")

## Cell 15 - Dashboards: one page per track
Six panels: (1) top-down map coloured by the class label, hollow = coasted packet; (2) altitude & speed; (3) lateral load & vertical rate;
(4) belief per class over time (stacked = 100 %), dashed line = class change; (5) confidence, margin and P[air];
(6) **physics gate**: how plausible each class is for the current altitude / speed / manoeuvre (dark = impossible).
Panel 6 is the clearest way to explain to colleagues *why* a boat can never be reported for a target at 1000 m.

In [ ]:
def plot_dashboard(key, show=True):
    S = stream[stream.track == key].sort_values("n").reset_index(drop=True); n = len(S)
    B = np.stack(S.cb.values); FE = np.stack(S.feas.values); t = S.t_rel.to_numpy() / 60; last = S.iloc[-1]
    fig = plt.figure(figsize=(17, 13)); gs = fig.add_gridspec(3, 2, hspace=0.38, wspace=0.18)
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(S.x / 1e3, S.y / 1e3, color="#bbbbbb", lw=1, zorder=1)
    ax.scatter(S.x / 1e3, S.y / 1e3, c=[COLORS.get(c, "#ccc") for c in S.stable], s=34, zorder=2,
               edgecolors=["k" if c else "none" for c in S.coast], linewidths=0.8)
    ax.scatter([0], [0], marker="^", s=110, color="k", zorder=3); ax.annotate("radar", (0, 0), textcoords="offset points", xytext=(6, 6), fontsize=8)
    ax.annotate("start", (S.x.iloc[0] / 1e3, S.y.iloc[0] / 1e3), textcoords="offset points", xytext=(6, 6), fontsize=8)
    ax.annotate("now", (S.x.iloc[-1] / 1e3, S.y.iloc[-1] / 1e3), textcoords="offset points", xytext=(6, -12), fontsize=8)
    ax.set_xlabel("x [km]"); ax.set_ylabel("y [km]"); ax.set_aspect("equal", "datalim"); ax.grid(alpha=.3); ax.set_title("Track (colour = class label, hollow = coasted)")
    ax = fig.add_subplot(gs[0, 1]); ax.plot(t, S.alt, color="#1f77b4", lw=2); ax.set_ylabel("altitude [m]", color="#1f77b4"); ax.set_xlabel("minutes since first packet")
    ax2 = ax.twinx(); ax2.plot(t, S.speed, color="#d62728", lw=2); ax2.set_ylabel("speed [m/s]", color="#d62728"); ax.grid(alpha=.3); ax.set_title("Altitude and speed")
    ax = fig.add_subplot(gs[1, 0]); ax.plot(t, S.lat_g, color="#9467bd", lw=2); ax.set_ylabel("lateral load [g]", color="#9467bd"); ax.set_xlabel("minutes since first packet")
    ax2 = ax.twinx(); ax2.plot(t, S.vz, color="#2ca02c", lw=1.5); ax2.set_ylabel("vertical speed [m/s]", color="#2ca02c"); ax.grid(alpha=.3); ax.set_title("Manoeuvre signature")
    ax = fig.add_subplot(gs[1, 1]); ax.stackplot(S.n, B.T, colors=[COLORS[c] for c in CLASSES], labels=CLASSES, alpha=0.92)
    for e in [e for e in S.events if e]: ax.axvline(e["pkt"], color="k", ls="--", lw=1.4); ax.text(e["pkt"], 1.02, f"{e['to'][:6]}", fontsize=7, ha="center")
    if n >= 2: ax.axvspan(0.5, min(MIN_PKTS, n) - 0.5, color="white", alpha=0.35, hatch="//", lw=0)
    ax.set_xlim(1, max(n, 2)); ax.set_ylim(0, 1); ax.set_xlabel("packet #"); ax.set_ylabel("belief"); ax.set_title("Class belief over time (hatched = provisional)")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=8, frameon=False)
    ax = fig.add_subplot(gs[2, 0]); ax.plot(S.n, S.conf, color="#000", lw=2, label="confidence"); ax.plot(S.n, S.margin, color="#ff7f0e", lw=1.5, label="margin to runner-up")
    ax.plot(S.n, S.p_air, color="#1f77b4", lw=1.5, ls="--", label="P[air]"); ax.axhline(CONF_OK, color="#2ca02c", ls=":", lw=1); ax.axhline(CONF_MIN, color="#d62728", ls=":", lw=1)
    ax.scatter(S.n[S.ood], np.full(S.ood.sum(), 1.03), marker="v", color="#d62728", label="OOD packet", s=18)
    ax.set_ylim(0, 1.08); ax.set_xlabel("packet #"); ax.set_ylabel("probability"); ax.grid(alpha=.3); ax.legend(fontsize=8, ncol=2); ax.set_title("Confidence (dotted: CONFIRMED / AMBIGUOUS thresholds)")
    ax = fig.add_subplot(gs[2, 1]); im = ax.imshow(np.log10(FE.T), aspect="auto", cmap="viridis", vmin=-4, vmax=0, extent=[0.5, n + 0.5, K - 0.5, -0.5])
    ax.set_yticks(range(K), [c.replace("_", " ").title() for c in CLASSES], fontsize=8); ax.set_xlabel("packet #"); ax.set_title("Physics gate: log10 plausibility of each class")
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="log10 feasibility (0 = fully plausible)")
    fig.suptitle(f"Track {key}:  {last.domain} -> {last.verdict}  [{last.status}, {last.conf:.0%}]   |   {n} packets", fontsize=15, fontweight="bold", y=0.995)
    if show: plt.show()
    return fig

for key in stream.track.unique()[:6]:
    plot_dashboard(key)

## Cell 16 - "Why this class?" envelope plot, and the class timeline of all tracks
**Left**: every dot is a simulated training track (colour = true class) in the speed-altitude plane; your real tracks are drawn
on top in black. This is the plot to show a colleague: a track at 1000 m / 190 m/s sits in the air-vehicle clouds and nowhere near
the vessel cluster at the bottom. **Right**: class timeline (bars) for each track; a colour change is a class change.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(17, 6.8), gridspec_kw=dict(width_ratios=[1.15, 1]))
ax = axs[0]
for i, c in enumerate(CLASSES):
    pts = np.array([[np.median(t["phys"][:, 1]), np.median(t["phys"][:, 0])] for t in train_tracks if t["labels"][-1] == c])
    ax.scatter(pts[:, 0], pts[:, 1], s=14, alpha=0.35, color=COLORS[c], label=c, linewidths=0)
for j, (key, S) in enumerate(stream.groupby("track", sort=False)):
    ax.plot(S.speed, S.alt, "-", color="k", lw=1.2, alpha=0.8); ax.scatter(S.speed, S.alt, s=14, color="k", zorder=5)
    ax.annotate(f"{key}", (S.speed.iloc[-1], S.alt.iloc[-1]), textcoords="offset points", xytext=(7, 5), fontsize=9, fontweight="bold")
ax.set_xscale("symlog", linthresh=10); ax.set_yscale("symlog", linthresh=50); ax.set_xlim(0, 2000); ax.set_ylim(0, 3e4)
ax.set_xlabel("speed [m/s]"); ax.set_ylabel("altitude [m]"); ax.grid(alpha=.3, which="both"); ax.legend(fontsize=8, ncol=3, loc="lower right", frameon=True)
ax.set_title("Where the real tracks sit among the simulated classes")
ax = axs[1]; keys = list(stream.track.unique())[:30]
for r, key in enumerate(keys):
    S = stream[stream.track == key].sort_values("n")
    for a, b, c in segments(S["stable"].tolist()): ax.barh(r, b - a + 1, left=a + 1, color=COLORS.get(c, "#ccc"), edgecolor="white", height=0.7)
ax.set_yticks(range(len(keys)), [k[:22] for k in keys], fontsize=8); ax.invert_yaxis(); ax.set_xlabel("packet #"); ax.grid(alpha=.3, axis="x")
ax.set_title("Class timeline per track"); handles = [plt.Rectangle((0, 0), 1, 1, color=COLORS[c]) for c in CLASSES + ["UNDETERMINED"]]
ax.legend(handles, CLASSES + ["UNDETERMINED"], fontsize=7, ncol=2, loc="upper right", bbox_to_anchor=(1.0, -0.12), frameon=False)
plt.tight_layout(); plt.show()

## Cell 17 - Self-tests (run these whenever you change anything)
1. **Sky-is-not-sea tests** on hand-made tracks, including the nasty case of only 1-3 packets (your `crossing_2` situation).
2. **Deployment equals training**: streaming a held-out simulated track packet by packet gives the same verdict as the offline path.
3. Your real files: any track above 300 m altitude must come out as AIR.

In [ ]:
def synth_icd(alt, speed, n, dt=10.0, seed=0, track="T", sea_heave=0.0, turn_g=0.0):
    r = np.random.default_rng(seed); t = np.arange(n) * dt
    om = turn_g * G0 / max(speed, 1.0); h = om * t
    vx, vy = speed * np.cos(h) + r.normal(0, 0.4, n), speed * np.sin(h) + r.normal(0, 0.4, n)
    x = 30000 + np.cumsum(vx) * dt; y = 5000 + np.cumsum(vy) * dt
    z = alt + sea_heave * np.sin(t / 8) + r.normal(0, 4, n)
    ax_, ay_ = (np.gradient(vx, dt), np.gradient(vy, dt)) if n > 1 else (np.zeros(n), np.zeros(n))
    return pd.DataFrame(dict(track_no=track, pos_x=x, pos_y=y, pos_z=z, vel_x=vx, vel_y=vy, vel_z=r.normal(0, 0.3, n),
                             accel_x=ax_ * 0.3, accel_y=ay_ * 0.3, accel_z=r.normal(0, 0.02, n), ntp_sec=(1.7e9 + t).astype(np.int64),
                             ntp_frac=0, assoc_indication=1))

def run_stream(df):
    c = LiveClassifier(); res = [c.update(r) for r in df.to_dict("records")]; return res[-1]

tests = [("airborne 1000 m, 190 m/s, 3 packets (crossing_2 case)", synth_icd(1000, 190, 3), "AIR"),
         ("airborne 1000 m, 190 m/s, 1 packet", synth_icd(1000, 190, 1), "AIR"),
         ("airborne 5000 m, 300 m/s, 60 packets", synth_icd(5000, 300, 60), "AIR"),
         ("airborne 300 m, 40 m/s (helicopter-like), 20 packets", synth_icd(300, 40, 20), "AIR"),
         ("airborne 12000 m, 240 m/s (airliner-like), 40 packets", synth_icd(12000, 240, 40), "AIR"),
         ("ship 15 m, 6 m/s, 40 packets", synth_icd(15, 6, 40, sea_heave=0.8), "SURFACE"),
         ("boat 4 m, 12 m/s, 30 packets", synth_icd(4, 12, 30, sea_heave=0.5, turn_g=0.05), "SURFACE"),
         ("ship 10 m, 8 m/s, 2 packets", synth_icd(10, 8, 2), "SURFACE")]
rows = []
for name, df, want in tests:
    r = run_stream(df); rows.append(dict(test=name, expected=want, got=r["domain"], P_air=round(r["p_air"], 4), verdict=r["verdict"], status=r["status"],
                                           result="PASS" if r["domain"] == want else "FAIL"))
display(pd.DataFrame(rows))

# 2) streaming == offline on a held-out simulated track
tr = iid_tracks[7]; c = LiveClassifier(); live = [c.update(r) for r in tr["raw"].to_dict("records")]
off = run_belief(tr, window_probs(model, make_windows([tr], stride_late=1)[0]), BELIEF)
diff = np.abs(np.stack([r["cb"] for r in live]) - calib(off)).max()
print(f"streaming vs offline: max belief difference {diff:.4f}  ->", "PASS" if diff < 0.02 else "CHECK")

# 3) real files
for key, S in stream.groupby("track", sort=False):
    if S.alt.median() > 300:
        ok = S.p_air.iloc[-1] > 0.95
        print(f"real track {key}: median altitude {S.alt.median():.0f} m -> P[air] = {S.p_air.iloc[-1]:.3f}  ", "PASS" if ok else "FAIL")

## Cell 18 - Save the bundle, and how to use it later
The bundle (model, scaling, temperatures, fusion parameters, physics envelopes) lets you classify **new files without
retraining**: run Cell 2 (imports/settings), Cell 3-5 (definitions), Cell 8-11 (functions, skip the training lines) is the
long way. The short way: keep this notebook, set `INPUT_FILES = [...]` in Cell 2 and re-run Cells 12-17 in the same session.

In [ ]:
model.save(ARTIFACT_DIR / "radar_classifier.keras")
joblib.dump(dict(FMEAN=FMEAN, FSTD=FSTD, TEMP=TEMP, TEMP_B=TEMP_B, BELIEF=BELIEF, ENVELOPE=ENVELOPE, CLASSES=CLASSES, SEQ_LEN=SEQ_LEN,
                 FEATURES=FEATURES, OOD_MU=OOD_MU, OOD_INV=OOD_INV, OOD_THR=OOD_THR), ARTIFACT_DIR / "radar_bundle.pkl")
print("saved:", sorted(p.name for p in ARTIFACT_DIR.iterdir()))
print("""
To reuse without retraining (new session): define Cells 2-6 functions (or import them from a .py), then
    B = joblib.load('radar_bundle_v20/radar_bundle.pkl'); model = keras.models.load_model('radar_bundle_v20/radar_classifier.keras', compile=False)
    FMEAN, FSTD, TEMP, TEMP_B, BELIEF = B['FMEAN'], B['FSTD'], B['TEMP'], B['TEMP_B'], B['BELIEF']
and run Cells 11-17.  Edit ENVELOPE (Cell 2) if a real platform legitimately exceeds a limit.""")

## Cell 19 - Research dashboard: setup (Plotly, dark / light theme)
Same visual language as your Model-11 plots (dark navy, interactive, exportable) with more on top: a synchronised
**mission replay animation**, class-coloured **3-D track**, **physics-gate** heatmap, **speed-altitude envelope**, tracker-quality
view, benchmark charts and a **single-file HTML report** you can open in any browser (no Python needed) for a meeting.
Every chart is also written to `radar_figures/*.html`. Hover for values, click legend entries to hide/show classes, drag to zoom.
Set `PLOT_THEME = "light"` for printing.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
PLOT_THEME = "dark"                      # "dark" (demos) or "light" (papers / printing)
FOCUS_TRACK = None                       # None = track with most packets, or e.g. "radar_data#1"
FIG_DIR = Path("radar_figures"); FIG_DIR.mkdir(exist_ok=True)
_TH = dict(dark=dict(bg="#0B1220", panel="#101B2D", grid="#263449", text="#E6EDF7", muted="#9FB0C8", accent="#48DDF5", start="#5CE0A3", end="#FF6B7A", tpl="plotly_dark"),
           light=dict(bg="#FFFFFF", panel="#FFFFFF", grid="#DCE3EB", text="#1E293B", muted="#64748B", accent="#087EA4", start="#16865B", end="#C83E4D", tpl="plotly_white"))[PLOT_THEME]
PAL = {"CIVILIAN_AIRCRAFT": "#55B7E8", "FIGHTER_AIRCRAFT": "#FF7185", "HELICOPTER": "#5CE0A3", "DRONE": "#FFBE69", "SURFACE_MISSILE": "#B69CFF",
       "AERIAL_MISSILE": "#FF8A65", "SKIFF": "#45D6C5", "BOAT": "#7BA7FF", "SHIP": "#B0BAC8", "UNDETERMINED": "#5B6B82"}
if PLOT_THEME == "light":
    PAL.update({"CIVILIAN_AIRCRAFT": "#1F77B4", "FIGHTER_AIRCRAFT": "#D62728", "HELICOPTER": "#2CA02C", "DRONE": "#FF7F0E", "SURFACE_MISSILE": "#7E57C2",
                "AERIAL_MISSILE": "#E4572E", "SKIFF": "#17A398", "BOAT": "#3F6FD8", "SHIP": "#6B7280", "UNDETERMINED": "#A0AEC0"})
SAVED_FIGS = {}


def style(fig, title, height=520, legend_h=True):
    fig.update_layout(template=_TH["tpl"], paper_bgcolor=_TH["bg"], plot_bgcolor=_TH["panel"], height=height, title=dict(text=title, x=0.01, xanchor="left"),
                      font=dict(color=_TH["text"], family="Inter, Arial, sans-serif"), margin=dict(l=60, r=30, t=115, b=55), hovermode="closest",
                      legend=dict(orientation="h" if legend_h else "v", bgcolor="rgba(0,0,0,0)", yref="container", y=(1 - 55 / height) if legend_h else 0.9, yanchor="top", x=0 if legend_h else 1.01))
    fig.update_xaxes(gridcolor=_TH["grid"], zerolinecolor=_TH["grid"]); fig.update_yaxes(gridcolor=_TH["grid"], zerolinecolor=_TH["grid"])
    return fig


def show_save(fig, name):
    fig.write_html(FIG_DIR / f"{name}.html", include_plotlyjs="cdn"); SAVED_FIGS[name] = fig; fig.show(); return fig


def focus_rows(key=None):
    key = key or FOCUS_TRACK or stream.groupby("track").size().idxmax()
    return key, stream[stream.track == key].sort_values("n").reset_index(drop=True)

focus_key, FS = focus_rows()
print(f"theme {PLOT_THEME} | focus track {focus_key} ({len(FS)} packets) | tracks available: {list(stream.track.unique())}")

### Plot 1 - Mission replay (animated)
Press **Play** or drag the slider. Left: the track so far (dots = class label, hollow ring = coasted packet). Top-right: the live
class belief. Bottom-right: confidence over time with a moving cursor. The title shows the verdict at that packet.

In [ ]:
def fig_replay(S, max_frames=140):
    n = len(S); B = np.stack(S.cb.values); idx = np.unique(np.linspace(0, n - 1, min(n, max_frames)).astype(int))
    col = [PAL.get(c, PAL["UNDETERMINED"]) for c in S.stable]; x, y = S.x / 1e3, S.y / 1e3
    fig = make_subplots(rows=2, cols=2, specs=[[{"rowspan": 2}, {}], [None, {}]], column_widths=[0.58, 0.42], row_heights=[0.55, 0.45],
                        horizontal_spacing=0.09, vertical_spacing=0.14, subplot_titles=("Track (top-down, km)", "Class belief now", "Confidence over time"))
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", line=dict(color=_TH["muted"], width=1, dash="dot"), opacity=0.45, name="recorded path", hoverinfo="skip"), 1, 1)
    fig.add_trace(go.Scatter(x=x[:1], y=y[:1], mode="markers", marker=dict(size=7, color=col[:1]), name="trail", hoverinfo="skip"), 1, 1)
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers+text", marker=dict(symbol="triangle-up", size=13, color=_TH["text"]), text=["radar"], textposition="top right", name="radar", hoverinfo="skip"), 1, 1)
    fig.add_trace(go.Scatter(x=x[:1], y=y[:1], mode="markers", marker=dict(size=17, color=col[:1], line=dict(color=_TH["bg"], width=2)), name="now"), 1, 1)
    fig.add_trace(go.Bar(x=B[0], y=CLASSES, orientation="h", marker_color=[PAL[c] for c in CLASSES], name="belief", showlegend=False,
                         hovertemplate="%{y}: %{x:.1%}<extra></extra>"), 1, 2)
    fig.add_trace(go.Scatter(x=S.n, y=S.conf, mode="lines", line=dict(color=_TH["accent"], width=3), name="confidence", hovertemplate="packet %{x}<br>%{y:.0%}<extra></extra>"), 2, 2)
    fig.add_trace(go.Scatter(x=S.n, y=S.p_air, mode="lines", line=dict(color=_TH["muted"], width=1.5, dash="dash"), name="P[air]", hovertemplate="packet %{x}<br>P[air] %{y:.1%}<extra></extra>"), 2, 2)
    fig.add_trace(go.Scatter(x=[1, 1], y=[0, 1], mode="lines", line=dict(color=_TH["end"], width=2), name="cursor", hoverinfo="skip"), 2, 2)
    frames = []
    for i in idx:
        r = S.iloc[i]
        frames.append(go.Frame(name=str(i), traces=[1, 3, 4, 7], data=[
            go.Scatter(x=x[:i + 1], y=y[:i + 1], mode="markers", marker=dict(size=7, color=col[:i + 1], line=dict(color=["white" if c else "rgba(0,0,0,0)" for c in S.coast[:i + 1]], width=1.2))),
            go.Scatter(x=[x[i]], y=[y[i]], mode="markers", marker=dict(size=17, color=[col[i]], line=dict(color=_TH["bg"], width=2))),
            go.Bar(x=B[i], y=CLASSES, orientation="h", marker_color=[PAL[c] for c in CLASSES]),
            go.Scatter(x=[r.n, r.n], y=[0, 1], mode="lines", line=dict(color=_TH["end"], width=2))],
            layout=go.Layout(title=dict(text=f"Packet {r.n}/{n}  |  {r.domain}  |  {r.verdict}  |  {r.conf:.0%}  {r.status}  |  alt {r.alt:.0f} m  v {r.speed:.0f} m/s"))))
    fig.frames = frames
    steps = [dict(method="animate", label=str(int(S.n.iloc[i])), args=[[str(i)], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for i in idx]
    fig.update_layout(updatemenus=[dict(type="buttons", showactive=False, x=0.0, y=-0.12, yanchor="top", xanchor="left", buttons=[
        dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=110, redraw=True), transition=dict(duration=0), fromcurrent=True, mode="immediate")]),
        dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])])],
        sliders=[dict(active=0, x=0.12, y=-0.12, yanchor="top", len=0.85, pad=dict(t=0, b=0), currentvalue=dict(prefix="Packet: ", font=dict(color=_TH["text"])), steps=steps)])
    style(fig, f"Mission replay - track {S.track.iloc[0]}", height=700); fig.update_layout(showlegend=False, margin=dict(l=60, r=30, t=70, b=150))
    fig.update_xaxes(title_text="x [km]", row=1, col=1); fig.update_yaxes(title_text="y [km]", scaleanchor="x", scaleratio=1, row=1, col=1)
    fig.update_xaxes(range=[0, 1], tickformat=".0%", row=1, col=2); fig.update_yaxes(autorange="reversed", row=1, col=2)
    fig.update_xaxes(title_text="packet #", row=2, col=2); fig.update_yaxes(range=[0, 1.02], tickformat=".0%", row=2, col=2)
    return fig

fig = show_save(fig_replay(FS), "01_mission_replay")

### Plot 2 - Class belief over time (stacked, with class-change markers)
Each colour band is the belief in one class; they always add to 100 %. Dashed lines mark class changes (hover the label for the cause).
The grey strip at the start is the provisional zone (fewer than 4 packets). Lower panel: confidence and margin to the runner-up.

In [ ]:
def fig_belief(S):
    n = len(S); B = np.stack(S.cb.values)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.06)
    for i, c in enumerate(CLASSES):
        fig.add_trace(go.Scatter(x=S.n, y=B[:, i], mode="lines", stackgroup="b", name=c, line=dict(width=0.6, color=PAL[c]), fillcolor=PAL[c],
                                 hovertemplate=f"{c}<br>packet %{{x}}: %{{y:.1%}}<extra></extra>"), 1, 1)
    for e in [e for e in S.events if e]:
        fig.add_vline(x=e["pkt"], line=dict(color=_TH["text"], dash="dash", width=1.4), row=1, col=1)
        fig.add_annotation(x=e["pkt"], y=1.0, yref="y domain", text=f"{e['frm'][:5]}->{e['to'][:5]}", showarrow=False, yshift=12, font=dict(size=10), hovertext=e["trigger"], row=1, col=1)
    fig.add_vrect(x0=0.5, x1=min(MIN_PKTS, n) + 0.5 - 1, fillcolor=_TH["muted"], opacity=0.25, line_width=0, row=1, col=1)
    fig.add_trace(go.Scatter(x=S.n, y=S.conf, mode="lines", name="confidence", line=dict(color=_TH["accent"], width=3)), 2, 1)
    fig.add_trace(go.Scatter(x=S.n, y=S.margin, mode="lines", name="margin to runner-up", line=dict(color=PAL["DRONE"], width=2)), 2, 1)
    fig.add_hline(y=CONF_OK, line=dict(color=_TH["start"], dash="dot", width=1), row=2, col=1); fig.add_hline(y=CONF_MIN, line=dict(color=_TH["end"], dash="dot", width=1), row=2, col=1)
    style(fig, f"Class belief & confidence - track {S.track.iloc[0]}", height=620)
    fig.update_yaxes(range=[0, 1], tickformat=".0%", row=1, col=1); fig.update_yaxes(range=[0, 1.02], tickformat=".0%", row=2, col=1)
    fig.update_xaxes(title_text="packet #", row=2, col=1, rangeslider=dict(visible=True, thickness=0.06))
    return fig
fig = show_save(fig_belief(FS), "02_belief_timeline")

### Plot 3 - 3-D track coloured by class
Rotate with the mouse. Dots = packets (colour = class label), thin vertical lines drop to the ground plane, the grey shadow is the ground track.

In [ ]:
def fig_3d(S):
    x, y, z = S.x, S.y, S.alt; txt = [f"packet {r.n}<br>{r.verdict}<br>{r.conf:.0%} {r.status}<br>alt {r.alt:.0f} m, v {r.speed:.0f} m/s" for r in S.itertuples()]
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=x, y=y, z=np.zeros(len(S)), mode="lines", line=dict(color=_TH["muted"], width=2), opacity=0.5, name="ground track", hoverinfo="skip"))
    fig.add_trace(go.Scatter3d(x=x, y=y, z=z, mode="lines", line=dict(color=_TH["accent"], width=4), name="path", hoverinfo="skip"))
    for c in [c for c in CLASSES + ["UNDETERMINED"] if (S.stable == c).any()]:
        m = (S.stable == c).to_numpy()
        fig.add_trace(go.Scatter3d(x=x[m], y=y[m], z=z[m], mode="markers", marker=dict(size=4.5, color=PAL[c]), name=c, text=np.array(txt)[m], hovertemplate="%{text}<extra></extra>"))
    dx, dy, dz = [], [], []
    for i in range(0, len(S), max(1, len(S) // 25)): dx += [x[i], x[i], None]; dy += [y[i], y[i], None]; dz += [0, z[i], None]
    fig.add_trace(go.Scatter3d(x=dx, y=dy, z=dz, mode="lines", line=dict(color=_TH["muted"], width=1), opacity=0.35, name="altitude drops", hoverinfo="skip"))
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers+text", text=["radar"], marker=dict(size=6, color=_TH["text"], symbol="diamond"), name="radar"))
    fig.add_trace(go.Scatter3d(x=[x.iloc[0]], y=[y.iloc[0]], z=[z.iloc[0]], mode="markers", marker=dict(size=7, color=_TH["start"]), name="start"))
    fig.add_trace(go.Scatter3d(x=[x.iloc[-1]], y=[y.iloc[-1]], z=[z.iloc[-1]], mode="markers", marker=dict(size=8, color=_TH["end"], symbol="diamond"), name="latest"))
    style(fig, f"3-D track - {S.track.iloc[0]}", height=680, legend_h=False); fig.update_layout(margin=dict(l=0, r=0, t=50, b=0), legend=dict(yref="paper", y=0.98, x=0.01, xanchor="left", yanchor="top"), scene_domain=dict(x=[0.12, 1], y=[0, 1]))
    fig.update_layout(scene=dict(xaxis_title="x [m]", yaxis_title="y [m]", zaxis_title="altitude [m]", aspectmode="manual", aspectratio=dict(x=1.4, y=1.0, z=0.55), camera=dict(eye=dict(x=1.35, y=-1.45, z=0.85)), bgcolor=_TH["panel"],
                                 xaxis=dict(gridcolor=_TH["grid"]), yaxis=dict(gridcolor=_TH["grid"]), zaxis=dict(gridcolor=_TH["grid"])))
    return fig
fig = show_save(fig_3d(FS), "03_trajectory_3d")

### Plot 4 - Kinematics with the class timeline behind it
Altitude, speed, lateral load and vertical speed on one shared time axis; the coloured background is the class label at that time.
This is the "evidence" view: you can *see* the manoeuvre that made the model change its mind.

In [ ]:
def fig_kin(S):
    fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.035, subplot_titles=("Altitude [m]", "Speed [m/s]", "Lateral load [g]", "Vertical speed [m/s]"))
    for r, (col, cc) in enumerate([("alt", "#5CE0A3"), ("speed", "#FFBE69"), ("lat_g", "#B69CFF"), ("vz", _TH["accent"])], start=1):
        fig.add_trace(go.Scatter(x=S.n, y=S[col], mode="lines", line=dict(color=cc, width=2.4), name=col, showlegend=False,
                                 hovertemplate="packet %{x}<br>%{y:.2f}<extra></extra>"), r, 1)
    for a, b, c in segments(S["stable"].tolist()):
        fig.add_vrect(x0=a + 0.5, x1=b + 1.5, fillcolor=PAL.get(c, PAL["UNDETERMINED"]), opacity=0.13, line_width=0)
    for c in [c for c in CLASSES + ["UNDETERMINED"] if (S.stable == c).any()]:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color=PAL[c], symbol="square"), name=c), 1, 1)
    style(fig, f"Kinematic evidence - {S.track.iloc[0]}", height=720); fig.update_xaxes(title_text="packet #", row=4, col=1)
    return fig
fig = show_save(fig_kin(FS), "04_kinematics")

### Plot 5 - Physics gate (why a boat is impossible at 1000 m)
Colour = plausibility of each class for the current altitude / speed / manoeuvre (bright = plausible, dark = physically excluded).

In [ ]:
def fig_gate(S):
    FE = np.stack(S.feas.values)
    fig = go.Figure(go.Heatmap(z=np.log10(FE.T), x=S.n, y=CLASSES, zmin=-4, zmax=0, colorscale="Viridis", colorbar=dict(title="log10<br>plausibility"),
                               hovertemplate="packet %{x}<br>%{y}<br>log10 plausibility %{z:.2f}<extra></extra>"))
    style(fig, f"Physics gate - {S.track.iloc[0]}", height=470); fig.update_yaxes(autorange="reversed"); fig.update_xaxes(title_text="packet #")
    return fig
fig = show_save(fig_gate(FS), "05_physics_gate")

### Plot 6 - Speed-altitude envelope: where your real tracks sit among the simulated classes
Click legend entries to isolate a class. Log axes. Your real tracks are the white lines with markers. Presentation-ready "why this class".

In [ ]:
def fig_envelope():
    fig = go.Figure()
    for c in CLASSES:
        pts = np.array([[np.median(t["phys"][:, 1]), np.median(t["phys"][:, 0])] for t in train_tracks if t["labels"][-1] == c])
        fig.add_trace(go.Scatter(x=pts[:, 0] + 1, y=pts[:, 1] + 1, mode="markers", name=c, marker=dict(size=6, color=PAL[c], opacity=0.5), hovertemplate=f"{c}<br>speed %{{x:.0f}} m/s<br>alt %{{y:.0f}} m<extra></extra>"))
    for key, S in stream.groupby("track", sort=False):
        fig.add_trace(go.Scatter(x=S.speed + 1, y=S.alt + 1, mode="lines+markers", name=f"REAL {key}", line=dict(color="white" if PLOT_THEME == "dark" else "black", width=2.5), marker=dict(size=6),
                                 text=[f"packet {i}" for i in S.n], hovertemplate=f"{key}<br>%{{text}}<br>speed %{{x:.0f}} m/s<br>alt %{{y:.0f}} m<extra></extra>"))
    style(fig, "Speed-altitude envelope: simulated classes vs. real tracks", height=600, legend_h=False)
    fig.update_xaxes(type="log", title_text="speed [m/s]"); fig.update_yaxes(type="log", title_text="altitude [m]")
    return fig
fig = show_save(fig_envelope(), "06_envelope")

### Plot 7 - All tracks: map + class timeline
For multi-track files. Click a track in the legend to hide it. Bars on the right show the class label over packets for each track.

In [ ]:
def fig_overview():
    keys = list(stream.track.unique())[:40]
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], subplot_titles=("Tracks (top-down, km)", "Class timeline"))
    for j, key in enumerate(keys):
        S = stream[stream.track == key].sort_values("n"); lc = PAL.get(S.stable.iloc[-1], PAL["UNDETERMINED"])
        fig.add_trace(go.Scatter(x=S.x / 1e3, y=S.y / 1e3, mode="lines+markers", name=str(key), legendgroup=str(key), line=dict(width=2.2, color=lc), marker=dict(size=4),
                                 text=[f"packet {i}: {v}" for i, v in zip(S.n, S.verdict)], hovertemplate=f"{key}<br>%{{text}}<extra></extra>"), 1, 1)
        for a, b, c in segments(S["stable"].tolist()):
            fig.add_trace(go.Bar(x=[b - a + 1], y=[str(key)], base=[a + 1], orientation="h", marker=dict(color=PAL.get(c, PAL["UNDETERMINED"]), line=dict(color=_TH["bg"], width=1)),
                                 legendgroup=str(key), showlegend=False, hovertemplate=f"{key}<br>{c}<br>packets {a + 1}-{b + 1}<extra></extra>"), 1, 2)
    fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers", marker=dict(symbol="triangle-up", size=12, color=_TH["text"]), name="radar"), 1, 1)
    style(fig, "All tracks", height=max(560, 150 + 26 * len(keys))); fig.update_layout(barmode="overlay")
    fig.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1); fig.update_yaxes(autorange="reversed", row=1, col=2); fig.update_xaxes(title_text="packet #", row=1, col=2)
    return fig
fig = show_save(fig_overview(), "07_all_tracks")

### Plot 8 - Track quality vs. confidence (where `std_dev` / `cov` are used - as diagnostics, not as class evidence)
The tracker's own position uncertainty (log scale) with coasted packets and out-of-distribution packets marked, next to the confidence.
A confidence dip that coincides with a coast or an uncertainty jump is a *data-quality* effect, not a behaviour change.

In [ ]:
def fig_quality(S):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.55, 0.45], vertical_spacing=0.07)
    fig.add_trace(go.Scatter(x=S.n, y=S.sig_pos, mode="lines", name="tracker position sigma [m]", line=dict(color=PAL["DRONE"], width=2.4)), 1, 1)
    c = S[S.coast]; fig.add_trace(go.Scatter(x=c.n, y=c.sig_pos, mode="markers", name="coasted packet", marker=dict(size=9, symbol="x", color=_TH["end"])), 1, 1)
    fig.add_trace(go.Scatter(x=S.n, y=S.conf, mode="lines", name="confidence", line=dict(color=_TH["accent"], width=3)), 2, 1)
    o = S[S.ood]; fig.add_trace(go.Scatter(x=o.n, y=o.conf, mode="markers", name="out-of-distribution packet", marker=dict(size=9, symbol="triangle-down", color=_TH["end"])), 2, 1)
    style(fig, f"Track quality - {S.track.iloc[0]}", height=520); fig.update_yaxes(type="log", title_text="sigma [m]", row=1, col=1)
    fig.update_yaxes(range=[0, 1.02], tickformat=".0%", row=2, col=1); fig.update_xaxes(title_text="packet #", row=2, col=1)
    return fig
fig = show_save(fig_quality(FS), "08_track_quality")

## Cell 20 - Benchmark charts (interactive versions of the evaluation)
These are the "does it actually work?" plots for a manager / reviewer. All numbers come from **unseen simulated tracks**
(familiar sensor and a harsher "stress" sensor). Remember: this measures the model against the simulator, not against reality -
the real-data gap is addressed in the roadmap section at the end.

In [ ]:
SHORT = {c: c.replace("_AIRCRAFT", "").replace("SURFACE_MISSILE", "SURF.MSL").replace("AERIAL_MISSILE", "AIR.MSL").replace("HELICOPTER", "HELI") for c in CLASSES}

def fig_confusions():
    fig = make_subplots(rows=1, cols=2, subplot_titles=[f"Familiar sensor - {np.mean(fv_iid.true == fv_iid.pred):.1%}", f"Stress sensor - {np.mean(fv_st.true == fv_st.pred):.1%}"], horizontal_spacing=0.12)
    for j, fv in enumerate([fv_iid, fv_st], 1):
        cm = confusion_matrix(fv.true, fv.pred, labels=CLASSES); sup = cm.sum(1)
        pct = np.divide(cm, sup[:, None], out=np.zeros(cm.shape), where=sup[:, None] > 0) * 100
        fig.add_trace(go.Heatmap(z=pct, x=[SHORT[c] for c in CLASSES], y=[SHORT[c] for c in CLASSES], colorscale="Blues", zmin=0, zmax=100, showscale=(j == 2),
                                 text=[[f"{v:.0f}" if v >= 1 else "" for v in r] for r in pct], texttemplate="%{text}", textfont=dict(size=11),
                                 customdata=cm, hovertemplate="true %{y}<br>predicted %{x}<br>%{z:.0f}% (%{customdata} tracks)<extra></extra>",
                                 colorbar=dict(title="% of true class", len=0.8)), 1, j)
        fig.update_yaxes(autorange="reversed", title_text="true" if j == 1 else None, row=1, col=j); fig.update_xaxes(title_text="predicted", row=1, col=j, tickangle=-40)
    return style(fig, "Final verdict per track - row-normalised confusion matrices", height=560)

def fig_per_class():
    rows = []
    for nm, fv in [("familiar", fv_iid), ("stress", fv_st)]:
        for c in CLASSES:
            t = fv[fv.true == c]; p = fv[fv.pred == c]
            rows.append(dict(split=nm, cls=c, recall=np.mean(t.pred == c) if len(t) else np.nan, precision=np.mean(p.true == c) if len(p) else np.nan))
    D = pd.DataFrame(rows); fig = make_subplots(rows=1, cols=2, subplot_titles=("Recall (of the real X, how many were found)", "Precision (when it says X, how often right)"), horizontal_spacing=0.08)
    for j, met in enumerate(["recall", "precision"], 1):
        for nm, col in [("familiar", _TH["accent"]), ("stress", _TH["end"])]:
            d = D[D.split == nm]; fig.add_trace(go.Bar(x=[SHORT[c] for c in d.cls], y=d[met], name=nm, marker_color=col, legendgroup=nm, showlegend=(j == 1),
                                                       hovertemplate="%{x}<br>%{y:.0%}<extra>" + nm + "</extra>"), 1, j)
        fig.update_yaxes(range=[0, 1.02], tickformat=".0%", row=1, col=j)
    fig.update_layout(barmode="group"); return style(fig, "Per-class performance (track-level final verdict)", height=470)

def fig_acc_vs_n():
    fig = go.Figure()
    for nm, tr_, sq_, col in [("familiar", iid_tracks, seq_iid, _TH["accent"]), ("stress", stress_tracks, seq_st, _TH["end"])]:
        xs, a_f, a_r, cnt = acc_by_n(tr_, sq_); ok = cnt > 20
        fig.add_trace(go.Scatter(x=xs[ok], y=a_f[ok], name=f"{nm}: fused (final)", line=dict(color=col, width=3), hovertemplate="%{x} packets: %{y:.0%}<extra>" + nm + " fused</extra>"))
        fig.add_trace(go.Scatter(x=xs[ok], y=a_r[ok], name=f"{nm}: raw network", line=dict(color=col, width=1.5, dash="dot"), hovertemplate="%{x} packets: %{y:.0%}<extra>" + nm + " raw</extra>"))
    fig.add_vrect(x0=0.5, x1=MIN_PKTS - 0.5, fillcolor=_TH["muted"], opacity=0.15, line_width=0, annotation_text="provisional", annotation_position="top left")
    fig.update_xaxes(title_text="packets received on the track"); fig.update_yaxes(title_text="exact-class accuracy", range=[0, 1.02], tickformat=".0%")
    return style(fig, "How quickly does it lock on? Accuracy vs. packets received", height=470)

def calib_points():
    conf, hit = [], []
    for tr, ps in zip(iid_tracks + stress_tracks, seq_iid + seq_st):
        B = calib(run_belief(tr, ps, BELIEF)); conf.append(B.max(1)); hit.append(B.argmax(1) == label_ids(tr))
    return np.concatenate(conf), np.concatenate(hit)

def fig_calibration():
    conf, hit = calib_points(); edges = np.linspace(0.1, 1.0, 10); idx = np.digitize(conf, edges) - 1; bx, by, bn = [], [], []
    for b in range(9):
        s = idx == b
        if s.sum() > 30: bx.append(conf[s].mean()); by.append(hit[s].mean()); bn.append(int(s.sum()))
    ece = float(np.sum(np.abs(np.array(bx) - np.array(by)) * np.array(bn)) / np.sum(bn))
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], subplot_titles=(f"Reliability diagram (ECE {ece:.3f})", "How often each confidence level is reported"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color=_TH["muted"], dash="dash"), name="perfect", hoverinfo="skip"), 1, 1)
    fig.add_trace(go.Scatter(x=bx, y=by, mode="lines+markers", marker=dict(size=np.clip(np.sqrt(bn) / 3, 7, 26), color=_TH["start"]), line=dict(color=_TH["start"], width=2),
                             name="model", customdata=bn, hovertemplate="said %{x:.0%} -> right %{y:.0%} (%{customdata} packets)<extra></extra>"), 1, 1)
    fig.add_trace(go.Histogram(x=conf, nbinsx=30, marker_color=_TH["accent"], name="reported confidence", showlegend=False), 1, 2)
    fig.update_xaxes(title_text="reported confidence", tickformat=".0%", row=1, col=1); fig.update_yaxes(title_text="observed accuracy", tickformat=".0%", range=[0, 1.02], row=1, col=1)
    fig.update_xaxes(title_text="reported confidence", tickformat=".0%", row=1, col=2)
    return style(fig, "Calibration - when it says 80 %, is it right 80 % of the time?", height=470)

def disguise_latencies():
    lat_true, lat_any = [], []
    for tr, ps in zip(iid_tracks + stress_tracks, seq_iid + seq_st):
        bo = tr["meta"]["breakout_idx"]
        if tr["meta"]["mimic"] and bo >= 2:
            B = run_belief(tr, ps, BELIEF).argmax(1); y = label_ids(tr)
            h = np.where(B[bo:] == y[bo])[0]; lat_true.append(int(h[0]) if len(h) else 99)
            c = np.where(B[bo:] != B[bo - 1])[0]; lat_any.append(int(c[0]) if len(c) else 99)
    return np.array(lat_any), np.array(lat_true)

def fig_disguise():
    la, lt = disguise_latencies(); ks = np.arange(0, 31)
    fig = go.Figure()
    for nm, L, col in [("verdict changes at all", la, _TH["accent"]), ("verdict = true class", lt, _TH["start"])]:
        fig.add_trace(go.Scatter(x=ks, y=[np.mean(L <= k) for k in ks], name=nm, mode="lines", line=dict(color=col, width=3, shape="hv"),
                                 hovertemplate="within %{x} packets: %{y:.0%}<extra>" + nm + "</extra>"))
    fig.update_xaxes(title_text="packets after the disguise ends"); fig.update_yaxes(title_text="share of disguised tracks unmasked", range=[0, 1.02], tickformat=".0%")
    return style(fig, f"Disguise unmasking speed ({len(la)} disguised test tracks)", height=450)

def fig_tiers():
    T = pd.concat([tab_iid, tab_st]).reset_index().rename(columns={"index": "stage"}); fig = go.Figure()
    for tier, col in [("domain", "#5CE0A3"), ("group", "#FFBE69"), ("class", "#FF7185")]:
        fig.add_trace(go.Bar(x=[f"{s[:11]} | {st}" for s, st in zip(T.split.str.replace("unseen tracks, ", ""), T.stage)], y=T[tier], name=tier, marker_color=col, hovertemplate="%{x}<br>" + tier + " %{y:.1%}<extra></extra>"))
    fig.update_layout(barmode="group"); fig.update_yaxes(range=[0.5, 1.01], tickformat=".0%")
    return style(fig, "Three accuracy tiers: domain (air/sea) > group > exact class", height=470)

BENCH = {}
for nm, f in [("09_confusion", fig_confusions), ("10_per_class", fig_per_class), ("11_accuracy_vs_packets", fig_acc_vs_n), ("12_calibration", fig_calibration),
              ("13_disguise_unmasking", fig_disguise), ("14_accuracy_tiers", fig_tiers)]:
    BENCH[nm] = show_save(f(), nm)

## Cell 21 - Disguise unmasking demo (animated)
Pick a simulated test track that *pretends* to be another class and then breaks out. The animation shows the class belief packet by
packet next to the speed / lateral-load evidence. The dotted line is the real moment the disguise ends - the lag between that line and the
belief flipping is the honest cost of "you cannot see a lie before it ends".

In [ ]:
def fig_unmask(track, ps, title=None):
    B = run_belief(track, ps, BELIEF); Bc = np.array([calib(b) for b in B]); n = len(B); bo = track["meta"]["breakout_idx"]; ph = track["phys"]
    mim, true_c = track["meta"]["mimic"], track["labels"][-1]
    fig = make_subplots(rows=2, cols=2, column_widths=[0.4, 0.6], row_heights=[0.5, 0.5], specs=[[{"rowspan": 2}, {}], [None, {}]], vertical_spacing=0.12, horizontal_spacing=0.1,
                        subplot_titles=("Class belief now", "Speed [m/s]", "Lateral load [g]"))
    fig.add_trace(go.Bar(x=Bc[0], y=CLASSES, orientation="h", marker_color=[PAL[c] for c in CLASSES]), 1, 1)
    fig.add_trace(go.Scatter(x=np.arange(1, n + 1), y=ph[:, 1], line=dict(color=_TH["accent"], width=2), showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=np.arange(1, n + 1), y=ph[:, 3], line=dict(color="#B69CFF", width=2), showlegend=False), 2, 2)
    fig.add_trace(go.Scatter(x=[1, 1], y=[0, ph[:, 1].max()], mode="lines", line=dict(color=_TH["end"], width=2), showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=[1, 1], y=[0, max(ph[:, 3].max(), 0.1)], mode="lines", line=dict(color=_TH["end"], width=2), showlegend=False), 2, 2)
    for r in (1, 2): fig.add_vline(x=bo + 1, line_dash="dot", line_color=_TH["start"], row=r, col=2)
    fig.add_annotation(x=bo + 1, y=1.02, xref="x2", yref="y2 domain", text="disguise ends", showarrow=False, font=dict(color=_TH["start"]))
    frames = []
    for k in range(n):
        top = CLASSES[int(Bc[k].argmax())]
        frames.append(go.Frame(name=str(k), traces=[0, 3, 4], data=[go.Bar(x=Bc[k], y=CLASSES, orientation="h", marker_color=[PAL[c] for c in CLASSES]),
                  go.Scatter(x=[k + 1, k + 1], y=[0, ph[:, 1].max()], mode="lines", line=dict(color=_TH["end"], width=2)),
                  go.Scatter(x=[k + 1, k + 1], y=[0, max(ph[:, 3].max(), 0.1)], mode="lines", line=dict(color=_TH["end"], width=2))],
                  layout=go.Layout(title=dict(text=f"Packet {k + 1}/{n} | verdict: {top} ({Bc[k].max():.0%}) | truth now: {track['labels'][k]}"))))
    fig.frames = frames
    steps = [dict(method="animate", label=str(k + 1), args=[[str(k)], dict(mode="immediate", frame=dict(duration=0, redraw=True), transition=dict(duration=0))]) for k in range(n)]
    fig.update_layout(updatemenus=[dict(type="buttons", showactive=False, x=0, y=-0.14, yanchor="top", xanchor="left", buttons=[
        dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=140, redraw=True), transition=dict(duration=0), fromcurrent=True, mode="immediate")]),
        dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate", transition=dict(duration=0))])])],
        sliders=[dict(active=0, x=0.12, y=-0.14, yanchor="top", len=0.85, currentvalue=dict(prefix="Packet: ", font=dict(color=_TH["text"])), steps=steps)])
    style(fig, title or f"Disguise demo: a {true_c} pretending to be a {mim}", height=640); fig.update_layout(showlegend=False, margin=dict(l=60, r=30, t=70, b=150))
    fig.update_xaxes(range=[0, 1], tickformat=".0%", row=1, col=1); fig.update_yaxes(autorange="reversed", row=1, col=1); fig.update_xaxes(title_text="packet #", row=2, col=2)
    return fig

def pick_demo(tracks, seqs):
    best = None
    for tr, ps in zip(tracks, seqs):
        bo = tr["meta"]["breakout_idx"]
        if tr["meta"]["mimic"] and bo >= 8 and len(tr["labels"]) - bo >= 12:
            B = run_belief(tr, ps, BELIEF).argmax(1); y = label_ids(tr)
            if (B[max(0, bo - 3):bo] == CLS_ID[tr["meta"]["mimic"]]).all() and (B[-3:] == y[-3:]).all(): best = (tr, ps); break
    return best or next((tr, ps) for tr, ps in zip(tracks, seqs) if tr["meta"]["mimic"] and tr["meta"]["breakout_idx"] >= 2)
_dt, _dp = pick_demo(iid_tracks + stress_tracks, seq_iid + seq_st)
BENCH["15_disguise_demo"] = show_save(fig_unmask(_dt, _dp), "15_disguise_demo")

## Cell 22 - One-file HTML report (open in any browser, send to your manager)
Bundles the mission summary table and every interactive chart into `radar_report.html`. The animations and hover tools keep working offline
if you set `OFFLINE_JS = True` (file gets ~4 MB larger because Plotly is embedded).

In [ ]:
OFFLINE_JS = False
def build_report(path="radar_report.html"):
    figs = {**SAVED_FIGS, **BENCH}; parts = []
    tbl = summary.drop(columns=[c for c in ["audit"] if c in summary.columns]).to_html(index=False, border=0, classes="tbl")
    first = True
    for name in sorted(figs):
        parts.append(f"<h2>{name[3:].replace('_', ' ').title()}</h2>" + figs[name].to_html(full_html=False, include_plotlyjs=("include" if OFFLINE_JS else "cdn") if first else False, auto_play=False,
                                                                                         default_width="100%")); first = False
    css = ("body{background:#0B1220;color:#E6EDF7;font-family:Inter,Arial,sans-serif;margin:0 auto;max-width:1280px;padding:24px}h1{font-weight:600}h2{margin-top:36px;border-bottom:1px solid #263449;padding-bottom:6px;font-weight:500}"
           ".tbl{border-collapse:collapse;font-size:13px}.tbl th{background:#16233a;text-align:left}.tbl td,.tbl th{padding:5px 10px;border-bottom:1px solid #263449}.note{color:#9FB0C8}")
    html = (f"<!doctype html><meta charset='utf-8'><title>Radar target classification report</title><style>{css}</style><h1>Radar target classification - mission report</h1>"
            f"<p class='note'>Accuracy figures refer to unseen <b>simulated</b> tracks; real-data accuracy must be measured on labelled real tracks.</p>"
            f"<h2>Mission summary</h2><div style='overflow-x:auto'>{tbl}</div>" + "".join(parts))
    Path(path).write_text(html, encoding="utf-8"); print(f"wrote {path} ({Path(path).stat().st_size / 1e6:.1f} MB) with {len(figs)} charts")
build_report()

## Cell 23 - Where to go next (evidence-based roadmap)
**What I tested before writing this** (same simulated data, same splits, unseen tracks, exact-class accuracy):

| Model | exact class (familiar / stress) | group | domain |
|---|---|---|---|
| CNN-BiGRU-attention (this notebook) | 70.5 % / 69.1 % | 93 % / 92 % | 99.9 % / 99.5 % |
| GRU only | 66.8 % / 64.1 % | 93 % / 91 % | 99.3 % / 98.6 % |
| Causal TCN | 72.2 % / 68.5 % | 94 % / 93 % | 99.9 % / 99.5 % |
| ExtraTrees on window statistics | 74.1 % / 71.4 % | 96 % / 94 % | 99.9 % / 99.7 % |
| Gradient boosting on window statistics | 74.3 % / 71.3 % | 96 % / 95 % | 99.9 % / 99.6 % |
| + "something just changed" delta features (disguise test) | 70.8 % / 70.0 % | - | - |

**Reading:** every architecture lands within about 5 points of each other, and a plain tree model is as good as the deep net. The ceiling is not the
architecture - it is that SKIFF / BOAT / SHIP, HELICOPTER / DRONE and FIGHTER / AIR-MISSILE have *overlapping definitions* in the simulator
(see the confusion matrices). Changing the network will not buy the next 10 points. Extra delta features for disguise detection helped only marginally.
Disguise: a target that behaves exactly like a civilian aircraft is, kinematically, a civilian aircraft until it stops; only the unmasking *speed* can be improved.

**What will actually make the project strong, in priority order**
1. **Measure the sim-to-real gap early.** Take any real tracks you have, compute the same features, and train a small classifier to tell simulated from real windows
   (classifier two-sample test). Accuracy near 50 % = simulator is realistic; high = see which features differ and fix the simulator (tracker noise, update rate, coast behaviour).
2. **Calibrate the simulator to real data** (approximate Bayesian computation over the tracker parameters: measurement noise, process noise, coast rate), so `std_dev` / `cov` and jitter match the real ICD output.
3. **Get even a small labelled real set** (a few hundred tracks) and fine-tune / validate on it. Nothing else beats this. Report real-data accuracy separately from simulated accuracy.
4. **Add cues that separate overlapping classes** if the ICD offers them: RCS, micro-Doppler / rotor signature, IFF / ESM flags, track-quality fields. Kinematics alone cannot split a drone from a helicopter.
5. **Redefine overlapping classes as hierarchy + attributes** (e.g. "small surface craft" with a size/speed attribute) - report the group when the class is genuinely ambiguous. The engine already does this with the AMBIGUOUS status.
6. **Masquerade-risk layer**: instead of one label, report P(true class is X | has behaved like Y for N packets) so disguises are flagged as *possible* before they break out.
7. **Then** consider self-supervised pre-training on unlabelled real tracks, and domain adaptation - they pay off only once real data exists.